# 📝 과제 02: 가위바위보 손 모양 맞히기와 과적합 규제

## 만들 결과

손 그림을 보고 바위, 보, 가위를 맞히는 CNN을 만듭니다. 정답을 붙인 학습 그림이 600장뿐인 상황에서 규제 없는 CNN의 과적합을 학습 곡선으로 확인하고, 교안 02의 규제를 하나씩 더해 검증 손실이 얼마나 낮아지는지 비교합니다.

<img src="images/86_rps_pipeline.png" width="1050" alt="제목은 '가위바위보 CNN: 과적합을 확인하고 규제로 줄이기'입니다. 번호 카드 여섯 장이 2행 3열로 화살표를 따라 이어집니다. 1 '손 모델로 나누기'에는 흑백 손 그림 '바위', '보', '가위'와 칩 '48 x 48', 파랑 칩 '학습 600장', 빨강 칩 '검증 720장'이 있습니다. 2 'CNN 만들기'에는 위에서 아래로 '(1, 48, 48)', '합성곱 + 풀링' 두 번, 'Linear', 주황 '로짓 3개' 상자가 이어집니다. 3 '과적합 확인'에는 가로축 '에폭' 그래프에서 파란 '학습 손실'은 바닥까지 내려가고 빨간 '검증 손실'은 앞쪽에서 가장 낮았다가 다시 올라갑니다. 4 '규제 비교'에는 '최저 검증 손실' 가로 막대 다섯 개 '규제 없음' 0.937, 'Dropout 0.5' 0.862, 'weight decay 2.0' 0.910, '증강' 0.346, '증강 + Dropout 0.5' 0.281이 있고 마지막 막대에 '고른 설정' 칩이 붙어 있습니다. 5 '조기 종료와 Lightning'에는 검증 손실 곡선의 가장 낮은 점에 초록 별과 그 오른쪽 'patience 10' 칩, 그 아래 'LightningModule', 'Trainer' 상자가 있습니다. 6 '테스트 정확도'에는 '테스트 372장' 칩과 보라 막대 두 개 '조기 종료' 0.866, 'Lightning' 0.839가 있습니다.">

- 손 모델로 나눈 학습 600장, 검증 720장, 테스트 372장과 48x48 입력에 맞춘 CNN
- 그림 단위 무작위 분할과 묶음 단위 분할의 검증, 테스트 정확도 비교(무작위 분할은 검증 정확도 0.951, 테스트 정확도 0.750)
- 규제 없는 CNN의 최저 검증 손실(0.937)과 50에폭의 검증 손실(1.805)
- Dropout, weight decay, 데이터 증강의 최저 검증 손실 비교표와 손 그림에 쓸 증강 고르기
- 고른 설정을 조기 종료와 PyTorch Lightning으로 학습한 두 모델의 테스트 정확도(약 0.84~0.87)

## 준비: 데이터와 제공 코드

- **자료:** `data/과제02/rps48.pt`. 컴퓨터 그래픽으로 그린 손 그림 데이터(Rock Paper Scissors, Laurence Moroney, CC BY 2.0)를 48x48 흑백으로 줄였습니다. `torch.load`로 읽으면 dict이고 키는 `train_images`(2,520장), `train_labels`, `test_images`(372장), `test_labels`, `class_names`입니다. 정답은 0=바위, 1=보, 2=가위입니다.
- **학습 그림의 순서:** 학습 그림은 손 모양(보, 바위, 가위) 순서로 840장씩 쌓여 있고, 그 안에서 손 모델(피부색과 손 모양이 다른 손) 7개가 120장씩 이어집니다. 같은 손 모델의 120장은 손을 조금씩 돌려 그린 그림이라 이웃한 그림끼리 거의 같습니다. 테스트 그림 372장은 학습 그림과 따로 내려받는 파일이고, 같은 그림이 학습 그림에 들어 있지 않습니다.
- **제공 코드:** 준비 셀이 아래 이름을 만듭니다.

| 이름 | 내용 |
|---|---|
| `rps` | 위 dict |
| `class_names` | `["바위", "보", "가위"]` |
| `hand_ids` | 학습 그림마다 손 모델 번호(0~6)를 담은 정수 텐서 |
| `train_one_epoch`, `evaluate` | 교안 02에서 만든 학습, 평가 함수(맨 앞에 `model.train()`, `model.eval()`이 있는 판) |
| `weight_size` | 교안 02에서 만든 가중치 크기 함수(절편 제외) |

- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행하세요. 뒤 문항이 앞 문항의 변수를 쓰므로 차례로 풉니다.
- **무작위로 고를 때의 정확도:** 종류가 3개라서 아무것도 배우지 못한 모델의 정확도는 1/3, 손실은 log 3 = 1.10 근처입니다.
- **실행 시간:** 학습 셀 하나는 CPU에서 20초~1분, 과제 전체는 CPU에 따라 3~10분 걸립니다.
- **실행 환경:** 계산 장치와 스레드 수에 따라 소수 셋째 자리와 멈춘 에폭이 달라질 수 있습니다. 결론은 곡선의 모양과 큰 차이로 읽습니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 가위바위보 손 그림: 학습 2,520장, 테스트 372장, 48 x 48 흑백
rps = torch.load(data_dir / "과제02" / "rps48.pt")
class_names = rps["class_names"]
print("학습 그림:", tuple(rps["train_images"].shape), "| 테스트 그림:", tuple(rps["test_images"].shape))
print("종류:", class_names)

# 손 모델 번호: 손 모양마다 840장이 손 모델 7개 x 120장으로 이어져 있습니다
# 그림 번호를 840으로 나눈 나머지가 손 모양 안에서의 순서이고, 그 값을 120으로 나눈 몫이 손 모델 번호입니다
hand_ids = (torch.arange(len(rps["train_images"])) % 840) // 120
print("손 모델별 그림 수:", torch.bincount(hand_ids).tolist())

In [ ]:
# 손 모양(행)마다 손 모델 7개(열)의 첫 그림
fig, axes = plt.subplots(3, 7, figsize=(14, 7.5))
for label in range(3):
    # 이 손 모양인 학습 그림의 번호(손 모델 0부터 120장씩 이어짐)
    label_index = torch.arange(len(rps["train_labels"]))[rps["train_labels"] == label]
    for hand in range(7):
        ax = axes[label, hand]
        ax.imshow(rps["train_images"][label_index[hand * 120]], cmap="gray")
        ax.set_title(f"{class_names[label]}, 손 모델 {hand}", fontsize=11)
        ax.axis("off")
plt.suptitle("손 모양과 손 모델별 학습 그림", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 같은 손 모델(보, 손 모델 0)의 이웃한 그림 6장: 손을 조금씩 돌렸을 뿐 거의 같습니다
fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for k in range(6):
    axes[k].imshow(rps["train_images"][k], cmap="gray")
    axes[k].set_title(f"그림 {k}", fontsize=11)
    axes[k].axis("off")
plt.suptitle("같은 손 모델의 이웃한 그림", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 교안 02에서 만든 학습, 평가, 가중치 크기, 에폭 간 변화 함수입니다. 실행만 하고 넘어갑니다.
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    model.train()  # 학습 모드: Dropout이 뉴런 출력 일부를 0으로 만듭니다
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    model.eval()  # 평가 모드: Dropout이 아무것도 0으로 만들지 않습니다
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples


def weight_size(model):
    """층 가중치(절편 제외)의 제곱을 모두 더해 제곱근을 씌운 값을 돌려줍니다."""
    total = 0.0
    for name, param in model.named_parameters():
        if name.endswith("weight"):
            total += (param ** 2).sum().item()
    return total ** 0.5


def epoch_change(history):
    """15에폭 뒤 구간에서 이웃한 두 에폭의 검증 손실 차이(절댓값)의 평균을 돌려줍니다."""
    # iloc[14:]는 15번째 행(15에폭)부터입니다. diff로 이웃한 에폭의 차이를 구해 절댓값을 평균합니다
    return history["검증 손실"].iloc[14:].diff().abs().mean()

## 1. 손 모델로 학습, 검증 데이터 나누기

**배경**: 이웃한 그림이 거의 같아서 그림 단위로 무작위로 나누면 검증 그림의 바로 옆 각도 그림이 학습에 들어가 검증 정확도가 부풉니다(교안 02 1-2의 데이터 누수). 그래서 손 모델 하나를 한 묶음으로 보고 묶음째 한쪽에만 넣는 **묶음 단위 분할(group split)** 로, 손 모델 5, 6은 검증 데이터로 떼고 손 모델 0~4의 1,800장 가운데 600장만 학습에 씁니다. 무작위로 나누면 검증 정확도가 얼마나 부푸는지는 3-4에서 확인합니다.

<img src="images/85_split_by_hand.png" width="1050" alt="제목은 '검증 데이터는 처음 보는 손으로'입니다. 위쪽 범례에 파랑 '학습', 빨강 '검증'이 있습니다. 왼쪽 판 '무작위로 나누기'에는 같은 손 카드 여섯 장의 테두리가 파랑, 빨강으로 번갈아 칠해져 있고, 이웃한 두 장마다 양쪽 화살표와 '거의 같은 그림'이 있으며, 아래에 '검증 정확도는 높지만 믿을 수 없음'이 있습니다. 오른쪽 판 '손 모델로 나누기'에는 피부색과 손 모양이 서로 다른 손 일곱 개가 서 있고, 밝은 손과 어두운 손이 학습과 검증 양쪽에 섞여 있습니다. 앞의 다섯 손에 '손 모델 0~4: 학습', 뒤의 두 손에 '손 모델 5, 6: 검증' 괄호가 붙어 있고, 아래에 '검증 = 처음 보는 손'이 있습니다.">

**요구사항**
- (1) 학습 그림 전체를 0~1 사이 float32로 바꾸고 채널 축을 더한 **`X_all`**(2520, 1, 48, 48)을 만듭니다.
- (2) 학습 그림의 정답 **`y_all`**(정수 2,520개)을 만듭니다.
- (3) 테스트 그림을 같은 방법으로 바꾼 **`X_test`**(372, 1, 48, 48)를 만듭니다.
- (4) 테스트 그림의 정답 **`y_test`**(정수 372개)를 만듭니다.
- (5) 손 모델 번호가 5 또는 6인 그림 번호 **`valid_index`**(720개)를 만듭니다.
- (6) 손 모델 번호가 0~4인 그림 번호 **`pool_index`**(1,800개)를 만듭니다.
- (7) `pool_index`를 시드 0 generator(`generator`, 제공 줄)로 섞어 앞 600개를 고른 **`train_index`** 를 만듭니다.
- (8) `X_all`, `y_all`에서 `train_index`로 고른 **`X_train`**, **`y_train`** 을 만듭니다.
- (9) `X_all`, `y_all`에서 `valid_index`로 고른 **`X_valid`**, **`y_valid`** 를 만듭니다.
- (10) 미니배치 크기 64인 DataLoader **`train_loader`**(섞기), **`valid_loader`**, **`test_loader`**(섞지 않기)를 만듭니다.

**예시**: 학습 데이터 600장은 손 모델 0~4에서 뽑았고, 검증 데이터 720장(손 모델 5, 6)은 학습에 한 장도 쓰지 않습니다.

**확인 기준**: 학습 600장, 검증 720장, 테스트 372장이고 `X_all`, `X_test`는 float32이며 값이 0~1입니다. 학습과 검증에 같은 손 모델이 없고, `train_index`가 시드 0으로 섞은 순서와 같으며, 학습 데이터에 세 손 모양이 고르게(종류마다 150장 넘게) 들어 있습니다. `train_loader`만 섞고 세 DataLoader의 미니배치 크기는 64입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 그림 번호(0~2519)를 담은 all_index를 손 모델 번호 조건으로 걸러 번호 목록을 만들고, 그 번호로 X_all, y_all을 인덱싱합니다.

세부구현:
1. 그림은 float로 바꿔 255로 나눈 뒤 unsqueeze로 채널 축을 더합니다.
2. 텐서에 조건을 넣으면 조건이 True인 자리만 고릅니다. 조건은 hand_ids와 비교 연산자로 만듭니다.
3. randperm의 개수는 pool_index의 길이이고 generator 인자는 제공 줄의 generator입니다. 섞인 순서로 pool_index를 인덱싱하고 앞 600개를 자릅니다.
4. 섞지 않고 앞 600개를 자르면 손 모양 하나(보)만 학습에 들어갑니다.
5. DataLoader에는 TensorDataset으로 묶은 그림과 정답, batch_size, shuffle을 줍니다. 학습용만 shuffle이 True입니다.
```

</details>

In [ ]:
# [작성] 손 모델로 학습, 검증 데이터를 나누고 DataLoader를 만드세요.
from torch.utils.data import DataLoader, TensorDataset

# 1. 픽셀을 0~1 실수로 바꾸고 채널 축 추가: (N, 48, 48) -> (N, 1, 48, 48)
X_all = ...
y_all = ...
X_test = ...
y_test = ...

# 2. 검증: 손 모델 5, 6의 그림 번호
all_index = torch.arange(len(X_all))
valid_index = ...

# 3. 학습: 손 모델 0~4의 그림 번호를 섞어 앞 600개
pool_index = ...
generator = torch.Generator().manual_seed(0)
train_index = ...

# 4. 학습 데이터와 검증 데이터
X_train = ...
y_train = ...
X_valid = ...
y_valid = ...

# 5. 미니배치: 학습용만 섞습니다
train_loader = ...
valid_loader = ...
test_loader = ...
print("학습:", tuple(X_train.shape), "| 검증:", tuple(X_valid.shape), "| 테스트:", tuple(X_test.shape))
print("학습 데이터의 종류별 그림 수:", torch.bincount(y_train).tolist())

In [ ]:
# [자가채점] 모양, 값의 범위, 손 모델 나누기, 미니배치를 확인합니다.
from torch.utils.data import RandomSampler

assert isinstance(X_all, torch.Tensor) and X_all.shape == (2520, 1, 48, 48), "X_all은 (2520, 1, 48, 48) 텐서입니다. `...`를 채웠는지, 채널 축을 더했는지 확인하세요."
assert X_all.dtype == torch.float32, "X_all은 float32입니다. 실수로 바꾼 뒤 255로 나누세요."
assert X_all.max().item() <= 1.0 and X_all.min().item() >= 0.0, "픽셀은 0~1 사이여야 합니다. 255로 나눴는지 확인하세요."
assert isinstance(y_all, torch.Tensor) and y_all.shape == (2520,), "y_all은 학습 그림의 정답 2,520개입니다."
assert isinstance(X_test, torch.Tensor) and X_test.shape == (372, 1, 48, 48) and X_test.max().item() <= 1.0, "X_test도 X_all과 같은 방법(실수로 바꾸기, 255로 나누기, 채널 축 더하기)으로 만듭니다."
assert isinstance(y_test, torch.Tensor) and y_test.shape == (372,), "y_test는 테스트 그림의 정답 372개입니다."

assert len(valid_index) == 720, f"valid_index는 손 모델 5, 6의 그림 720개입니다. 지금은 {len(valid_index)}개입니다."
assert (hand_ids[valid_index] >= 5).all(), "valid_index에는 손 모델 5, 6의 그림만 들어갑니다."
assert len(pool_index) == 1800 and (hand_ids[pool_index] <= 4).all(), "pool_index는 손 모델 0~4의 그림 번호 1,800개입니다."
assert len(train_index) == 600 and len(torch.unique(train_index)) == 600, "train_index는 서로 다른 그림 번호 600개입니다."
assert (hand_ids[train_index] <= 4).all(), "train_index에 손 모델 5, 6의 그림이 들어 있습니다. 전체를 섞지 말고 손 모델 0~4의 번호(pool_index)만 섞으세요."
train_counts = torch.bincount(y_all[train_index], minlength=3)
assert (train_counts > 150).all(), f"학습 데이터의 종류별 그림 수가 {train_counts.tolist()}입니다. 그림이 손 모양 순서로 쌓여 있으니 pool_index를 섞은 뒤 앞 600개를 고르세요."
assert train_index[:8].tolist() == [44, 1688, 1187, 1380, 1415, 584, 373, 2181] and train_index.sum().item() == 693808, "train_index가 시드 0으로 섞은 순서와 다릅니다. 제공 줄의 generator를 섞는 함수에 넘겼는지, pool_index를 섞었는지 확인하세요."

assert torch.equal(X_train, X_all[train_index]) and torch.equal(y_train, y_all[train_index]), "X_train, y_train은 X_all, y_all을 같은 train_index로 고른 것입니다. 그림과 정답을 같은 번호로 골랐는지 확인하세요."
assert torch.equal(X_valid, X_all[valid_index]) and torch.equal(y_valid, y_all[valid_index]), "X_valid, y_valid는 X_all, y_all을 같은 valid_index로 고른 것입니다."

assert train_loader.batch_size == 64 and valid_loader.batch_size == 64 and test_loader.batch_size == 64, "세 DataLoader 모두 미니배치 크기가 64입니다."
assert isinstance(train_loader.sampler, RandomSampler), "train_loader는 순서를 섞어야 합니다."
assert not isinstance(valid_loader.sampler, RandomSampler) and not isinstance(test_loader.sampler, RandomSampler), "valid_loader, test_loader는 섞지 않습니다. 평가는 순서를 섞을 필요가 없습니다."
assert len(train_loader.dataset) == 600 and len(valid_loader.dataset) == 720 and len(test_loader.dataset) == 372, "train_loader는 학습 600장, valid_loader는 검증 720장, test_loader는 테스트 372장을 담습니다."
print("✅ 1번 통과")

**결과 읽기**
- **학습 600장, 검증 720장, 테스트 372장:** 학습과 검증은 손 모델이 겹치지 않습니다.
- **종류별 199, 194, 207장:** 손 모델 0~4의 1,800장을 섞어 600장을 골랐으므로 세 손 모양이 비슷한 수로 들어 있습니다.

## 2. 48x48 그림에 맞는 CNN 만들기

교안 01의 CNN을 그대로 쓰되, 그림 크기가 28x28에서 48x48로 바뀌어 마지막 `nn.Linear`의 입력 크기가 달라집니다. 그 크기를 모양으로 확인한 뒤 Dropout 비율을 받는 CNN 함수를 만듭니다.

### 2-1. Flatten 뒤 크기 확인하기

**배경**: 교안 01은 28x28 그림이 풀링 두 번으로 7x7이 되었지만 48x48이면 Flatten 뒤 크기가 달라지므로, 그림을 한 층씩 통과시켜 모양을 출력해 확인합니다.

**요구사항**
- (1) 교안 01 CNN의 합성곱 부분(`Conv2d(1, 16)`, ReLU, `MaxPool2d(2)`, `Conv2d(16, 32)`, ReLU, `MaxPool2d(2)`, Flatten)을 `nn.Sequential`로 만든 **`conv_part`** 를 만듭니다. 합성곱은 둘 다 `kernel_size=3, padding=1`입니다.
- (2) 학습 그림 64장을 `conv_part`에 한 층씩 통과시키며 `층`(`type(layer).__name__`), `출력 모양` 두 키의 dict를 모은 표 **`shape_table`** 을 만듭니다. 첫 행은 골격에 적힌 입력 행입니다.
- (3) Flatten 뒤 그림 한 장의 크기(마지막 출력 모양의 둘째 값)를 정수 **`flat_size`** 로 만듭니다.

**예시**: 28x28 그림은 풀링 두 번 뒤 32 x 7 x 7 = 1,568개로 펼쳐집니다.

**확인 기준**: `conv_part`는 일곱 층이고 합성곱의 채널, `kernel_size`, `padding`과 풀링 크기가 위와 같습니다. `shape_table`은 입력 행과 층 7개로 8행이고, 마지막 행의 출력 모양 둘째 값이 `flat_size`입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01 4절에서 층마다 출력 모양을 표로 모은 방법과 같습니다.

세부구현:
1. nn.Sequential은 for 문으로 층을 하나씩 꺼낼 수 있고, 꺼낸 층에 텐서를 넣으면 그 층의 출력이 나옵니다.
2. 행 하나는 골격의 입력 행과 같은 두 키의 dict입니다.
3. 마지막 출력의 모양은 (64, Flatten 뒤 크기)입니다.
```

</details>

In [ ]:
# [작성] 합성곱 부분에 그림을 통과시켜 층마다 모양을 확인하세요.
import torch.nn as nn

# 1. 교안 01 CNN의 합성곱 부분(Flatten까지)
conv_part = ...

# 2. 학습 그림 64장을 한 층씩 통과시키며 출력 모양 기록
out = X_train[:64]
shape_rows = [{"층": "입력", "출력 모양": tuple(out.shape)}]
for layer in conv_part:
    out = ...
    shape_rows.append(...)
shape_table = pd.DataFrame(shape_rows)
display(shape_table)

# 3. Flatten 뒤 그림 한 장의 크기
flat_size = ...
print("Flatten 뒤 크기:", flat_size)

In [ ]:
# [자가채점] 합성곱 부분의 구조와 Flatten 뒤 크기를 확인합니다.
assert isinstance(conv_part, nn.Sequential) and len(conv_part) == 7, "conv_part는 Conv2d, ReLU, MaxPool2d, Conv2d, ReLU, MaxPool2d, Flatten 일곱 층의 nn.Sequential입니다."
assert isinstance(conv_part[0], nn.Conv2d) and conv_part[0].out_channels == 16 and conv_part[0].kernel_size == (3, 3) and conv_part[0].padding == (1, 1), "첫 층은 출력 채널 16, kernel_size 3, padding 1인 nn.Conv2d입니다."
assert isinstance(conv_part[3], nn.Conv2d) and conv_part[3].out_channels == 32 and conv_part[3].kernel_size == (3, 3) and conv_part[3].padding == (1, 1), "넷째 층은 출력 채널 32, kernel_size 3, padding 1인 nn.Conv2d입니다."
assert isinstance(conv_part[2], nn.MaxPool2d) and isinstance(conv_part[5], nn.MaxPool2d) and conv_part[2].kernel_size in (2, (2, 2)) and conv_part[5].kernel_size in (2, (2, 2)), "셋째, 여섯째 층은 크기 2인 nn.MaxPool2d입니다."
assert isinstance(conv_part[6], nn.Flatten), "마지막 층은 nn.Flatten입니다."
assert isinstance(shape_table, pd.DataFrame) and len(shape_table) == 8, "shape_table은 입력 행과 층 7개, 8행입니다. 반복문 안에서 층마다 한 행씩 더했는지 확인하세요."
assert list(shape_table["층"])[1:] == ["Conv2d", "ReLU", "MaxPool2d", "Conv2d", "ReLU", "MaxPool2d", "Flatten"], "층 칸에는 층의 클래스 이름을 넣습니다."
assert tuple(shape_table["출력 모양"].iloc[-1]) == (64, flat_size), "shape_table의 마지막 출력 모양은 (64, flat_size)입니다. 그림 64장을 넣었는지, flat_size에 마지막 출력 모양의 둘째 값을 담았는지 확인하세요."
print("✅ 2-1번 통과")

**결과 읽기**
- **(64, 16, 48, 48)에서 (64, 32, 12, 12):** 풀링이 한 번에 가로, 세로를 절반으로 줄여 48이 24, 12가 됩니다.
- **flat_size 4608:** 32 x 12 x 12 = 4,608이라 출력층 `nn.Linear`의 입력 크기는 28x28 그림의 1,568과 다릅니다.

### 2-2. Dropout 비율을 받는 CNN 함수 만들기

**배경**: 규제 없는 모델과 Dropout 모델을 같은 초깃값에서 비교하려고 Dropout 비율만 인자로 받는 함수를 만듭니다. `nn.Dropout(0.0)`은 아무 뉴런도 0으로 만들지 않아 Dropout이 없는 모델과 같습니다.

**요구사항**
- (1) **`make_rps_cnn(dropout_p)`** 는 골격대로 맨 앞에서 `torch.manual_seed(0)`으로 초깃값을 고정합니다.
- (2) 2-1의 합성곱 부분(Flatten까지) 뒤에 Dropout(비율 `dropout_p`)과 출력층(입력 `flat_size`, 출력 3)을 붙인 9층 `nn.Sequential`을 돌려줍니다.
- (3) 층은 함수 안에서 새로 만듭니다. 2-1의 `conv_part`를 넣으면 모델마다 같은 층(같은 파라미터)을 나눠 쓰게 됩니다.

**예시**: `make_rps_cnn(0.5)`의 여덟째 층은 `nn.Dropout`이고 비율이 0.5입니다.

**확인 기준**: 9층이고 여덟째 층의 Dropout 비율이 인자와 같으며 마지막 층이 입력 `flat_size`, 출력 3인 `nn.Linear`입니다. 두 번 만든 모델의 첫 층 가중치가 같고 2-1의 `conv_part` 층을 다시 쓰지 않습니다. 그림 5장을 넣으면 `(5, 3)` 로짓이 나오고 학습 전 검증 손실은 log 3 = 1.10 근처입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2-1의 nn.Sequential 일곱 층을 그대로 다시 적고, 그 뒤에 두 층을 더 적습니다.

세부구현:
1. 여덟째 층은 nn.Dropout에 인자로 받은 dropout_p를 넘깁니다.
2. 아홉째 층은 nn.Linear로, 입력 크기는 2-1의 flat_size, 출력 크기는 손 모양 수 3입니다.
```

</details>

In [ ]:
# [작성] Dropout 비율을 받는 CNN 함수를 완성하세요.
def make_rps_cnn(dropout_p):
    """합성곱 두 번과 Dropout, 출력층으로 된 가위바위보 CNN을 만듭니다."""
    # 1. 초깃값이 실행마다 같게 시드 고정
    torch.manual_seed(0)

    # 2. 합성곱 부분 -> Dropout -> 출력층(손 모양 3개의 로짓)
    model = ...
    return model.to(device)

In [ ]:
# [자가채점] 출력 모양, 초깃값 고정, Dropout 비율, 학습 전 손실을 확인합니다.
import math

check_a = make_rps_cnn(0.0)
check_b = make_rps_cnn(0.5)
assert isinstance(check_a, nn.Sequential) and len(check_a) == 9, "make_rps_cnn은 합성곱 부분 7층 + Dropout + Linear, 9층의 nn.Sequential입니다. `...`를 채웠는지 확인하세요."
assert isinstance(check_a[7], nn.Dropout) and check_a[7].p == 0.0 and check_b[7].p == 0.5, "여덟째 층은 nn.Dropout입니다. 인자로 받은 비율을 넣었는지 확인하세요."
assert isinstance(check_a[8], nn.Linear) and check_a[8].in_features == flat_size and check_a[8].out_features == 3, "마지막 층은 입력 flat_size, 출력 3인 nn.Linear입니다."
assert check_a[0] is not conv_part[0], "make_rps_cnn이 2-1의 conv_part 층을 다시 쓰고 있습니다. 함수 안에서 층을 새로 만드세요."
assert torch.equal(check_a[0].weight, check_b[0].weight), "두 번 만든 모델의 초깃값이 다릅니다. 함수 맨 앞에서 시드를 고정했는지 확인하세요."

check_a.eval()
with torch.no_grad():
    check_logits = check_a(X_valid[:5].to(device))
assert check_logits.shape == (5, 3), f"그림 5장의 출력 모양이 {tuple(check_logits.shape)}입니다. (5, 3)이어야 합니다."
first_loss, first_accuracy = evaluate(check_a, valid_loader, nn.CrossEntropyLoss())
assert 1.0 < first_loss < 1.25, f"학습 전 검증 손실이 {first_loss:.3f}입니다. 아직 배우지 못한 모델은 log 3 = 1.10 근처입니다."
print("✅ 2-2번 통과: 학습 전 검증 손실", round(first_loss, 3), "| log 3 =", round(math.log(3), 3))

**결과 읽기**
- **학습 전 검증 손실 1.099:** log 3 = 1.099와 비슷합니다. 학습 전 모델은 무작위로 고를 때와 같은 수준입니다.
- **(5, 3):** 그림 5장마다 바위, 보, 가위 세 로짓이 나옵니다.

## 3. 규제 없이 학습하고 과적합 진단하기

규제를 더하기 전에 규제 없는 CNN이 이 데이터에서 언제부터 과적합되는지 확인합니다. 이 학습 곡선이 4~7절 비교의 기준입니다.

### 3-1. 학습 함수 만들기

**배경**: 같은 학습을 설정만 바꿔 여러 번 하므로, 에폭마다 학습하고 검증 데이터로 평가하는 과정을 함수 하나로 묶습니다. 설정은 검증 데이터로 고르므로 테스트 데이터는 쓰지 않습니다.

**요구사항**
- (1) 골격의 `...` 두 곳을 채워 **`fit_rps(model, optimizer, epochs, use_augment=False)`** 를 완성합니다.
- (2) 에폭마다 `train_one_epoch`로 `train_loader`를 한 번 학습하고, `evaluate`로 `valid_loader`의 손실과 정확도를 구합니다.
- (3) `use_augment=True`일 때 쓰는 `train_one_epoch_augment`는 6절에서 만듭니다. 그 전까지는 `use_augment=False`로만 부릅니다.

**예시**: `fit_rps(model, optimizer, epochs=2)`는 `에폭`, `학습 손실`, `검증 손실`, `검증 정확도` 네 열의 2행 표를 돌려줍니다.

**확인 기준**: 2에폭만 학습하면 표가 2행이고 열 이름이 위와 같습니다. 학습 뒤 파라미터가 바뀌고, 마지막 행의 검증 손실과 검증 정확도가 학습을 마친 모델을 검증 그림 전체로 다시 평가한 값과 거의 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 fit과 같은 일을 합니다. 두 제공 함수의 인자 순서를 준비 셀에서 확인합니다.

세부구현:
1. train_one_epoch는 모델, 학습 DataLoader, 손실 함수, 옵티마이저를 받아 학습 손실 하나를 돌려줍니다.
2. evaluate는 모델, DataLoader, 손실 함수를 받아 손실과 정확도 두 값을 돌려줍니다. 여기서는 검증 DataLoader를 넘깁니다.
```

</details>

In [ ]:
# [작성] 에폭마다 학습하고 검증 데이터로 평가하는 함수를 완성하세요.
def fit_rps(model, optimizer, epochs, use_augment=False):
    """에폭마다 학습하고 학습 손실, 검증 손실, 검증 정확도를 기록한 표를 돌려줍니다."""
    # 1. 미니배치 섞는 순서와 Dropout이 고르는 뉴런이 실행마다 같게 시드 고정
    torch.manual_seed(0)
    loss_fn = nn.CrossEntropyLoss()

    rows = []
    for epoch in range(1, epochs + 1):
        # 2. 한 에폭 학습(use_augment가 True이면 6절의 증강 판)
        if use_augment:
            train_loss = train_one_epoch_augment(model, train_loader, loss_fn, optimizer)
        else:
            train_loss = ...

        # 3. 검증 데이터로 평가
        valid_loss, valid_accuracy = ...
        rows.append({"에폭": epoch, "학습 손실": train_loss, "검증 손실": valid_loss, "검증 정확도": valid_accuracy})

    return pd.DataFrame(rows)

In [ ]:
# [자가채점] 2에폭만 학습해 함수를 확인합니다.
check_model = make_rps_cnn(0.0)
check_before = check_model[0].weight.detach().clone()
check_history = fit_rps(check_model, torch.optim.Adam(check_model.parameters(), lr=0.001), epochs=2)
display(check_history.set_index("에폭").round(3))
assert isinstance(check_history, pd.DataFrame) and len(check_history) == 2, "fit_rps는 에폭마다 한 행인 표를 돌려줍니다. epochs가 2이면 2행입니다. `...`를 채웠는지 확인하세요."
assert list(check_history.columns) == ["에폭", "학습 손실", "검증 손실", "검증 정확도"], "표의 열은 에폭, 학습 손실, 검증 손실, 검증 정확도입니다."
assert not torch.equal(check_before, check_model[0].weight.detach()), "학습 뒤에도 파라미터가 그대로입니다. train_one_epoch에 모델, 학습 DataLoader, 손실 함수, 옵티마이저를 순서대로 넘겼는지 확인하세요."

# 학습을 마친 모델에 검증 그림 720장을 한 번에 넣어 손실과 정확도를 따로 계산해 비교합니다
check_model.eval()
with torch.no_grad():
    check_logits = check_model(X_valid.to(device)).cpu()
recheck_loss = nn.functional.cross_entropy(check_logits, y_valid).item()
recheck_accuracy = (check_logits.argmax(dim=1) == y_valid).float().mean().item()
assert abs(check_history["검증 손실"].iloc[-1] - recheck_loss) < 0.005, "마지막 행의 검증 손실이 검증 데이터로 다시 계산한 값과 다릅니다. evaluate에 valid_loader를 넘겼는지 확인하세요."
assert abs(check_history["검증 정확도"].iloc[-1] - recheck_accuracy) < 0.01, "검증 정확도는 evaluate가 돌려주는 두 값 가운데 둘째 값입니다."
print("✅ 3-1번 통과")

**결과 읽기**
- **학습 손실 1.082에서 0.985:** 2에폭 표의 학습 손실이 첫 에폭에서 둘째 에폭으로 줄었습니다.
- **검증 손실 1.115:** 같은 모델을 검증 그림 720장에 한 번에 넣어 다시 계산한 값과 같아, 표의 검증 칸이 `valid_loader` 결과임을 확인했습니다.

### 3-2. 규제 없이 50에폭 학습하기

**배경**: 규제를 더하기 전의 기준 곡선이 있어야 규제 효과를 비교할 수 있으므로, Dropout 없는 모델을 먼저 50에폭 학습합니다.

**요구사항**
- (1) Dropout 비율 0.0인 `make_rps_cnn` 모델 **`base_model`** 을 만듭니다.
- (2) `base_model`의 파라미터를 학습률 0.001로 갱신하는 Adam 옵티마이저 **`base_optimizer`** 를 만듭니다.
- (3) `fit_rps`로 50에폭 학습한 표 **`base_history`**(50행)를 만듭니다.

**예시**: 규제 없는 모델은 학습 데이터를 거의 외우므로 학습 손실이 0에 가까워지는 동안 검증 손실은 다시 오릅니다.

**확인 기준**: 표는 50행이고 `base_model`의 Dropout 비율이 0.0이며 `base_optimizer`가 학습률 0.001인 Adam입니다. 마지막 에폭의 학습 손실은 0.1보다 작고(학습 데이터는 거의 다 외움), 마지막 검증 손실은 가장 낮았던 검증 손실의 1.2배보다 큽니다.

<details><summary>힌트</summary>

```text
세부구현:
1. 옵티마이저는 교안 02처럼 torch.optim.Adam에 모델의 parameters와 lr을 넘겨 만듭니다.
2. fit_rps의 인자는 모델, 옵티마이저, 에폭 수 순서입니다.
```

</details>

In [ ]:
# [작성] 규제 없는 CNN을 50에폭 학습하세요.
base_model = ...
base_optimizer = ...
base_history = ...
display(base_history.set_index("에폭").tail().round(3))

In [ ]:
# [자가채점] 학습 손실과 검증 손실이 갈라지는지 확인합니다.
assert isinstance(base_history, pd.DataFrame) and len(base_history) == 50, "base_history는 50에폭의 표(50행)입니다. `...`를 채웠는지 확인하세요."
assert base_model[7].p == 0.0, "base_model은 규제 없는 모델(Dropout 비율 0.0)입니다."
assert isinstance(base_optimizer, torch.optim.Adam) and base_optimizer.param_groups[0]["lr"] == 0.001, "base_optimizer는 학습률 0.001인 torch.optim.Adam입니다."

base_last = base_history.iloc[-1]
assert base_last["학습 손실"] < 0.1, f"마지막 학습 손실이 {base_last['학습 손실']:.3f}입니다. 규제 없는 모델은 50에폭이면 학습 데이터를 거의 다 외워 0.1보다 작아집니다."
assert base_last["검증 손실"] > 1.2 * base_history["검증 손실"].min(), "마지막 검증 손실이 가장 낮았던 검증 손실보다 충분히 크지 않습니다. 검증 손실을 valid_loader로 구했는지 확인하세요."
print("✅ 3-2번 통과")

아래 셀은 실행만 하세요. 에폭별 학습 손실과 검증 손실을 그립니다.

In [ ]:
# 규제 없는 CNN의 학습 곡선
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(base_history["에폭"], base_history["학습 손실"], marker="o", markersize=3, label="학습 손실")
ax.plot(base_history["에폭"], base_history["검증 손실"], marker="o", markersize=3, label="검증 손실")
ax.axhline(math.log(3), color="gray", linestyle="--", linewidth=1, label="log 3")
ax.set_title("규제 없는 CNN의 학습 곡선(학습 600장)", fontsize=14)
ax.set_xlabel("에폭", fontsize=12)
ax.set_ylabel("손실", fontsize=12)
ax.tick_params(labelsize=11)
ax.legend(loc="center right", fontsize=11)  # 두 곡선이 위아래로 갈라진 오른쪽 가운데 빈자리에 둡니다
plt.tight_layout()
plt.show()

**결과 읽기**
- **50에폭의 학습 손실 0.003:** 학습 그림 600장을 거의 다 외웠습니다.
- **50에폭의 검증 손실 1.805:** 학습 손실이 줄어드는 동안 검증 손실은 오르며 log 3(1.10)보다 높아졌습니다.

### 3-3. 검증 손실이 가장 낮았던 에폭 찾기

**배경**: 검증 손실이 가장 낮은 에폭이 처음 보는 손을 가장 잘 맞힌 때이고, 그 뒤로 학습 손실은 줄어도 검증 손실이 오르면 과적합입니다.

**요구사항**
- (1) `base_history`에서 검증 손실이 가장 낮은 행 **`base_best_row`** 를 만듭니다.
- (2) 그 행의 에폭을 정수(`int`) **`base_best_epoch`** 로 만듭니다.
- (3) 그 행의 검증 손실 **`base_best_valid_loss`** 를 만듭니다.

**예시**: 검증 손실 열이 [1.0, 0.9, 0.95]이면 가장 낮은 행은 둘째 행이고 에폭은 2입니다.

**확인 기준**: `base_best_valid_loss`가 표의 검증 손실 가운데 가장 작은 값이고, `base_best_epoch`가 그 값이 있는 행의 에폭(정수)이며 마지막 에폭보다 앞입니다.

<details><summary>힌트</summary>

```text
세부구현:
1. 열의 idxmin은 가장 작은 값이 있는 행의 이름(인덱스)을 돌려줍니다.
2. 표의 loc에 행 이름을 주면 그 행을 꺼내고, 행에 열 이름을 주면 값을 읽습니다.
3. 꺼낸 행은 정수 열과 실수 열이 섞여 있어 에폭이 10.0처럼 실수로 나옵니다. int로 바꿉니다.
```

</details>

In [ ]:
# [작성] 검증 손실이 가장 낮았던 에폭을 찾으세요.
base_best_row = ...
base_best_epoch = ...
base_best_valid_loss = ...
print("검증 손실이 가장 낮은 에폭:", base_best_epoch, "| 검증 손실:", round(base_best_valid_loss, 3))
print("그 에폭의 검증 정확도:", round(base_best_row["검증 정확도"], 3))
print("마지막 에폭의 검증 손실:", round(base_history["검증 손실"].iloc[-1], 3))

In [ ]:
# [자가채점] 최저 검증 손실과 그 에폭을 확인합니다.
import numbers  # 파이썬 int와 numpy 정수를 함께 정수로 확인합니다

assert abs(base_best_valid_loss - base_history["검증 손실"].min()) < 1e-9, "base_best_valid_loss는 검증 손실 열의 가장 작은 값입니다. 학습 손실 열이나 가장 큰 값을 쓰지 않았는지 확인하세요."
best_check = base_history[base_history["에폭"] == base_best_epoch]["검증 손실"].iloc[0]
assert abs(best_check - base_best_valid_loss) < 1e-9, "base_best_epoch는 검증 손실이 가장 낮은 행의 에폭 값입니다. 행 번호(0부터)가 아니라 에폭 열의 값(1부터)을 쓰세요."
assert base_best_epoch < 50, "검증 손실이 가장 낮은 에폭이 마지막 에폭입니다. 3-2를 다시 확인하세요."
assert isinstance(base_best_epoch, numbers.Integral), "base_best_epoch는 정수로 바꾼 값입니다. 꺼낸 행의 에폭은 10.0처럼 실수로 나옵니다."
print("✅ 3-3번 통과")

**결과 읽기**
- **10에폭:** 검증 손실이 가장 낮았던 때(0.937)입니다. 이 에폭의 검증 정확도는 0.554입니다.
- **50에폭에는 1.805:** 가장 낮았던 값보다 높아졌고, 이 차이가 과적합의 크기입니다. 학습 600장의 손 모델 5개를 외웠을 뿐, 처음 보는 손 모델에는 점점 더 자신 있게 틀립니다.

### 3-4. 그림 단위 무작위 분할의 누수 확인하기

**배경**: 1번의 묶음 단위 분할 대신 학습 그림 2,520장을 그림 단위로 무작위로 나누면, 같은 손 모델의 이웃 그림이 학습과 검증에 함께 들어갑니다. 두 분할에서 같은 CNN을 10에폭 학습해 검증 정확도와 테스트 정확도를 비교합니다. 이 두 모델은 설정을 고르는 데 쓰지 않으므로, 검증 정확도가 처음 보는 그림의 정확도를 얼마나 그대로 보여 주는지 확인하려고 테스트 데이터로 한 번씩 평가합니다.

**요구사항**
- (1) 학습 그림 2,520장의 번호를 시드 0 generator(`leak_generator`, 제공 줄)로 섞은 순서 **`leak_order`** 를 만듭니다.
- (2) `leak_order`의 앞 600개 **`leak_train_index`** 를 만듭니다.
- (3) `leak_order`에서 그다음 720개 **`leak_valid_index`** 를 만듭니다.
- (4) `X_all`, `y_all`에서 `leak_train_index`로 고른 그림과 정답을 미니배치 64, 순서 섞기로 나눠 주는 **`leak_train_loader`** 를 만듭니다.
- (5) 같은 방법으로 `leak_valid_index`의 그림과 정답을 섞지 않고 나눠 주는 **`leak_valid_loader`** 를 만듭니다.
- (6) `split_loaders`(분할 이름 -> (학습 DataLoader, 검증 DataLoader), 골격에 있음)를 돌며, 분할마다 Dropout 비율 0.0인 `make_rps_cnn` 모델을 학습률 0.001인 Adam으로 그 분할의 학습 DataLoader로 10에폭 학습해 **`split_models`**(분할 이름 -> 모델)에 담습니다. 이 문항은 `fit_rps`를 쓰지 않습니다(`fit_rps`는 1번의 `train_loader`, `valid_loader`로만 학습합니다). 에폭마다 제공 함수 `train_one_epoch`에 그 분할의 학습 DataLoader를 넘깁니다.
- (7) 분할마다 `분할`, `검증 정확도`(그 분할의 검증 DataLoader), `테스트 정확도`(`test_loader`) 세 키의 dict를 더한 표 **`split_table`**(2행, `split_loaders` 순서)을 만듭니다.

**예시**: 그림 단위 무작위 분할에서는 손 모델 7개가 모두 학습과 검증 양쪽에 들어갑니다. 1번의 묶음 단위 분할은 두 쪽에 함께 들어간 손 모델이 없습니다.

**확인 기준**: `leak_train_index` 600개와 `leak_valid_index` 720개가 겹치지 않고 시드 0으로 섞은 순서와 같으며, 두 DataLoader는 미니배치 크기가 64이고 학습용만 섞습니다. 두 모델은 그 분할의 학습 그림을 0.92 넘게 맞히고(그 분할의 학습 DataLoader로 학습함), `split_table`의 정확도가 `split_models`의 모델을 그 분할의 검증 DataLoader와 `test_loader`로 다시 평가한 값과 같습니다. 무작위 분할은 검증 정확도가 테스트 정확도보다 0.1 넘게 높고, 그 차이가 묶음 단위 분할의 차이보다 큽니다.

<details><summary>힌트</summary>

```text
접근방법:
- 1번에서 pool_index를 섞어 자른 방법을 학습 그림 전체의 번호에 쓰고, 분할마다 그 분할의 DataLoader로 학습합니다.

세부구현:
1. randperm의 개수는 학습 그림 수이고 generator 인자는 제공 줄의 leak_generator입니다. 섞은 순서를 슬라이싱해 앞 600개와 그다음 720개를 자릅니다.
2. DataLoader는 1번처럼 TensorDataset으로 묶은 그림과 정답, batch_size, shuffle을 줍니다.
3. 분할마다 모델과 옵티마이저를 새로 만들고, 에폭마다 제공 함수 train_one_epoch로 그 분할의 학습 DataLoader를 학습합니다.
4. 정확도는 evaluate가 돌려주는 두 값 가운데 둘째 값입니다.
```

</details>

In [ ]:
# [작성] 그림 단위 무작위 분할과 묶음 단위 분할에서 같은 CNN을 학습해 검증, 테스트 정확도를 비교하세요.

# 1. 그림 단위 무작위 분할: 2,520장을 섞어 앞 600장은 학습, 그다음 720장은 검증
leak_generator = torch.Generator().manual_seed(0)
leak_order = ...
leak_train_index = ...
leak_valid_index = ...
leak_train_loader = ...
leak_valid_loader = ...

# 2. 분할 이름 -> (학습 DataLoader, 검증 DataLoader)(제공)
split_loaders = {
    "무작위 분할": (leak_train_loader, leak_valid_loader),
    "묶음 단위 분할": (train_loader, valid_loader),
}

# 3. 분할마다 규제 없는 CNN을 10에폭 학습하고, 검증 정확도와 테스트 정확도를 계산
loss_fn = nn.CrossEntropyLoss()
split_models = {}
split_rows = []
for split_name, (split_train_loader, split_valid_loader) in split_loaders.items():
    # 새 모델과 옵티마이저, 10에폭 학습
    ...
    split_models[split_name] = split_model

    # 검증 정확도와 테스트 정확도로 한 행
    ...
    split_rows.append(...)

split_table = pd.DataFrame(split_rows)
display(split_table.round(3))

In [ ]:
# [자가채점] 무작위 분할의 번호와 DataLoader, 두 분할의 검증, 테스트 정확도를 확인합니다.
assert isinstance(leak_order, torch.Tensor) and leak_order.shape == (2520,), "leak_order는 학습 그림 2,520장의 번호를 섞은 텐서입니다. `...`를 채웠는지 확인하세요."
assert torch.equal(torch.sort(leak_order).values, torch.arange(2520)), "leak_order에는 0~2519가 한 번씩 들어 있어야 합니다. 섞는 번호의 개수가 학습 그림 수와 같은지 확인하세요."
assert leak_order[:8].tolist() == [1124, 217, 1699, 366, 703, 1164, 487, 1529], "leak_order가 시드 0으로 섞은 순서와 다릅니다. 제공 줄의 leak_generator를 섞는 함수에 넘겼는지 확인하세요."
assert len(leak_train_index) == 600 and len(leak_valid_index) == 720, f"leak_train_index는 600개, leak_valid_index는 720개입니다. 지금은 {len(leak_train_index)}개, {len(leak_valid_index)}개입니다."
assert len(set(leak_train_index.tolist()) & set(leak_valid_index.tolist())) == 0, "leak_train_index와 leak_valid_index에 같은 그림 번호가 있습니다. 검증 번호는 학습 600개 다음부터 자르세요."
assert leak_train_index.sum().item() == 742936 and leak_valid_index.sum().item() == 903717, "leak_train_index, leak_valid_index가 leak_order의 앞 600개, 그다음 720개가 아닙니다."

# 무작위 분할은 같은 손 모델이 양쪽에 들어가고, 묶음 단위 분할은 겹치지 않습니다
leak_shared = set(hand_ids[leak_train_index].tolist()) & set(hand_ids[leak_valid_index].tolist())
group_shared = set(hand_ids[train_index].tolist()) & set(hand_ids[valid_index].tolist())
assert len(leak_shared) == 7 and len(group_shared) == 0, f"양쪽에 함께 든 손 모델이 무작위 분할 {len(leak_shared)}개, 묶음 단위 분할 {len(group_shared)}개입니다. 무작위 분할은 학습 그림 전체의 번호를 섞어야 합니다."

# DataLoader
assert leak_train_loader.batch_size == 64 and leak_valid_loader.batch_size == 64, "두 DataLoader 모두 미니배치 크기가 64입니다."
assert isinstance(leak_train_loader.sampler, RandomSampler) and not isinstance(leak_valid_loader.sampler, RandomSampler), "leak_train_loader만 순서를 섞습니다."
assert torch.equal(leak_train_loader.dataset.tensors[1], y_all[leak_train_index]) and torch.equal(leak_valid_loader.dataset.tensors[1], y_all[leak_valid_index]), "leak_train_loader에는 leak_train_index, leak_valid_loader에는 leak_valid_index로 고른 그림과 정답을 넣으세요."
assert torch.equal(leak_train_loader.dataset.tensors[0], X_all[leak_train_index]), "leak_train_loader의 그림이 leak_train_index로 고른 X_all과 다릅니다. 그림과 정답을 같은 번호로 고르세요."

# 두 분할의 모델과 표
assert list(split_models) == ["무작위 분할", "묶음 단위 분할"], "split_models에는 split_loaders의 두 분할 이름마다 학습한 모델을 담습니다."
assert split_models["무작위 분할"] is not split_models["묶음 단위 분할"], "분할마다 make_rps_cnn으로 새 모델을 만드세요."
for model_check in split_models.values():
    assert isinstance(model_check, nn.Sequential) and model_check[7].p == 0.0, "두 모델은 Dropout 비율 0.0인 make_rps_cnn 모델입니다."
assert isinstance(split_table, pd.DataFrame) and list(split_table.columns) == ["분할", "검증 정확도", "테스트 정확도"], "split_table의 열은 분할, 검증 정확도, 테스트 정확도입니다."
assert split_table["분할"].tolist() == list(split_loaders), "split_table은 split_loaders 순서의 2행입니다."

# 표의 정확도: 학생의 두 모델을 그 분할의 검증 DataLoader와 test_loader로 다시 평가해 비교합니다
for k, (split_name, (_, check_valid_loader)) in enumerate(split_loaders.items()):
    _, recheck_valid = evaluate(split_models[split_name], check_valid_loader, nn.CrossEntropyLoss())
    _, recheck_test = evaluate(split_models[split_name], test_loader, nn.CrossEntropyLoss())
    assert abs(split_table["검증 정확도"].iloc[k] - recheck_valid) < 0.01, f"{split_name}의 검증 정확도가 그 분할의 검증 DataLoader로 다시 평가한 값({recheck_valid:.3f})과 다릅니다."
    assert abs(split_table["테스트 정확도"].iloc[k] - recheck_test) < 0.01, f"{split_name}의 테스트 정확도가 test_loader로 다시 평가한 값({recheck_test:.3f})과 다릅니다. 테스트 정확도는 test_loader로 계산합니다."

# 분할마다 그 분할의 학습 데이터로 학습했는지: 10에폭이면 자기 학습 그림은 0.96 안팎을 맞힙니다
for split_name, (check_train_loader, _) in split_loaders.items():
    _, own_train_accuracy = evaluate(split_models[split_name], DataLoader(check_train_loader.dataset, batch_size=64), nn.CrossEntropyLoss())
    assert own_train_accuracy > 0.92, f"{split_name} 모델이 그 분할의 학습 그림을 {own_train_accuracy:.3f}만 맞힙니다. 반복문 안에서 split_train_loader(그 분할의 학습 DataLoader)로 학습했는지 확인하세요."

# 누수: 무작위 분할만 검증 정확도가 테스트 정확도보다 크게 부풉니다
leak_gap = split_table["검증 정확도"].iloc[0] - split_table["테스트 정확도"].iloc[0]
group_gap = split_table["검증 정확도"].iloc[1] - split_table["테스트 정확도"].iloc[1]
assert leak_gap > 0.1, f"무작위 분할의 검증 정확도가 테스트 정확도보다 {leak_gap:.3f} 높습니다. 0.1 넘게 높아야 합니다. 검증 정확도는 그 분할의 검증 DataLoader, 테스트 정확도는 test_loader로 계산했는지 확인하세요."
assert leak_gap > group_gap, "무작위 분할의 차이(검증 정확도 - 테스트 정확도)가 묶음 단위 분할보다 크지 않습니다. 두 분할의 이름과 DataLoader가 짝이 맞는지 확인하세요."
print(f"✅ 3-4번 통과: 검증 정확도 - 테스트 정확도 | 무작위 분할 {leak_gap:.3f} | 묶음 단위 분할 {group_gap:.3f}")

**결과 읽기**
- **무작위 분할의 검증 정확도 0.951, 테스트 정확도 0.750:** 검증 정확도가 0.201 부풀었습니다. 검증 그림의 바로 옆 각도 그림을 이미 학습했기 때문입니다.
- **묶음 단위 분할의 검증 정확도 0.572, 테스트 정확도 0.656:** 검증 정확도가 테스트 정확도보다 높게 나오지 않아, 처음 보는 손에서의 정확도를 부풀리지 않습니다.
- **테스트 정확도는 무작위 분할이 더 높음:** 무작위 분할 모델은 손 모델 7개를 모두 학습했습니다. 문제는 모델의 성능이 아니라 검증 정확도를 믿을 수 없다는 점이라, 설정은 묶음 단위 분할의 검증 데이터로 고릅니다.

## 4. Dropout 더하기

교안 02처럼 Flatten 뒤에 Dropout을 넣어, 출력층이 특정 특징 몇 개에만 기대지 않게 합니다.

### 4-1. 학습 모드와 평가 모드 확인하기

**배경**: Dropout은 학습 모드에서만 뉴런을 무작위로 0으로 만들므로, 같은 그림을 두 번 넣으면 학습 모드에서는 로짓이 달라지고 평가 모드에서는 같아야 합니다.

**요구사항**
- (1) Dropout 비율 0.5인 `make_rps_cnn` 모델 **`drop_model`** 을 만듭니다. 제공 줄이 처음에 평가 모드로 바꿔 둡니다.
- (2) `drop_model`을 학습 모드로 바꾼 뒤 검증 그림 8장을 두 번 넣어, 두 로짓 차이의 절댓값 가운데 가장 큰 값을 파이썬 숫자 **`train_diff`** 로 만듭니다.
- (3) 평가 모드로 바꾼 뒤 같은 방법으로 구한 **`eval_diff`** 를 만듭니다.

**예시**: 평가 모드에서는 같은 그림이 같은 로짓을 만들어 `eval_diff`가 0입니다.

**확인 기준**: `train_diff`는 0보다 크고 `eval_diff`는 0이며, 둘 다 파이썬 숫자입니다. `drop_model`의 Dropout 비율은 0.5입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 3절에서 모드를 바꾼 메서드 두 개를 첫 `...` 자리와 둘째 `...` 자리에 씁니다.

세부구현:
1. 두 로짓의 차이를 빼서 구하고, abs로 절댓값, max로 가장 큰 값을 고릅니다.
2. 결과는 숫자 하나인 텐서이므로 item으로 파이썬 숫자를 꺼냅니다.
3. 처음 모드는 평가 모드로 제공되므로 학습 모드로 바꾸는 줄을 빠뜨리면 train_diff가 0이 됩니다.
```

</details>

In [ ]:
# [작성] 학습 모드와 평가 모드에서 같은 그림을 두 번 넣어 로짓을 비교하세요.
drop_model = ...
drop_model.eval()  # 제공: 처음에는 평가 모드로 둡니다
sample = X_valid[:8].to(device)

# 1. 학습 모드: Dropout이 켜집니다
...  # drop_model을 학습 모드로 바꾸기
with torch.no_grad():
    logits_1 = drop_model(sample)
    logits_2 = drop_model(sample)
train_diff = ...

# 2. 평가 모드: Dropout이 꺼집니다
...  # drop_model을 평가 모드로 바꾸기
with torch.no_grad():
    logits_1 = drop_model(sample)
    logits_2 = drop_model(sample)
eval_diff = ...
print("학습 모드 두 번의 로짓 차이:", round(train_diff, 4), "| 평가 모드:", eval_diff)

In [ ]:
# [자가채점] 두 모드의 차이를 확인합니다.
assert isinstance(drop_model, nn.Sequential) and drop_model[7].p == 0.5, "drop_model은 Dropout 비율 0.5인 make_rps_cnn 모델입니다."
assert isinstance(train_diff, float) and isinstance(eval_diff, float), "train_diff, eval_diff는 파이썬 숫자입니다. 텐서에서 숫자를 꺼냈는지, `...`를 채웠는지 확인하세요."
assert train_diff > 0, "학습 모드에서 두 로짓이 같습니다. 첫 `...` 자리에서 drop_model을 학습 모드로 바꿔 Dropout을 켜세요."
assert eval_diff == 0, f"평가 모드에서 두 로짓이 {eval_diff:.4f}만큼 다릅니다. 둘째 `...` 자리에서 drop_model을 평가 모드로 바꿔 Dropout을 끄세요."
print("✅ 4-1번 통과")

**결과 읽기**
- **학습 모드 0.2254:** 같은 그림인데 두 로짓이 이만큼 다릅니다. 매번 다른 뉴런이 0이 되기 때문입니다.
- **평가 모드 0:** 평가 모드에서는 Dropout이 꺼져 같은 그림이 같은 로짓을 만듭니다.

### 4-2. Dropout 0.5로 50에폭 학습하기

**배경**: 4-1의 `drop_model`은 아직 학습하지 않아 초깃값이 `base_model`과 같으므로, 그대로 학습하면 Dropout만 다른 비교가 됩니다.

**요구사항**
- (1) `drop_model`의 파라미터를 학습률 0.001로 갱신하는 Adam 옵티마이저 **`drop_optimizer`** 를 만듭니다.
- (2) `fit_rps`로 50에폭 학습한 표 **`drop_history`**(50행)를 만듭니다.

**예시**: 3-2와 같은 방법이고 모델만 `drop_model`입니다.

**확인 기준**: 표는 50행이고 `drop_optimizer`가 `drop_model`의 파라미터를 학습률 0.001로 갱신하는 Adam입니다. 마지막 에폭의 검증 손실이 규제 없음(`base_history`)의 마지막 검증 손실보다 작습니다.

<details><summary>힌트</summary>

```text
세부구현:
1. 3-2와 같은 방법이고 모델만 drop_model입니다.
```

</details>

In [ ]:
# [작성] Dropout 0.5 모델을 50에폭 학습하세요.
drop_optimizer = ...
drop_history = ...
display(drop_history.set_index("에폭").tail().round(3))

In [ ]:
# [자가채점] Dropout 모델의 학습 결과를 확인합니다.
assert isinstance(drop_history, pd.DataFrame) and len(drop_history) == 50, "drop_history는 50에폭의 표(50행)입니다."
assert isinstance(drop_optimizer, torch.optim.Adam) and drop_optimizer.param_groups[0]["lr"] == 0.001, "drop_optimizer는 학습률 0.001인 torch.optim.Adam입니다."
assert drop_optimizer.param_groups[0]["params"][0] is drop_model[0].weight, "drop_optimizer는 drop_model의 파라미터로 만드세요."
drop_last = drop_history["검증 손실"].iloc[-1]
base_last = base_history["검증 손실"].iloc[-1]
assert drop_last < base_last, f"Dropout 모델의 마지막 검증 손실({drop_last:.3f})이 규제 없음({base_last:.3f})보다 작지 않습니다. drop_model의 Dropout 비율이 0.5인지 확인하세요."
print("✅ 4-2번 통과: 최저 검증 손실", round(drop_history["검증 손실"].min(), 3), "| 마지막 검증 손실", round(drop_last, 3))

**결과 읽기**
- **50에폭의 검증 손실 1.292:** 규제 없음(1.805)보다 낮습니다.
- **최저 검증 손실 0.862:** 규제 없음(0.937)보다 0.075 낮습니다. 이 차이가 에폭 간 변화보다 큰지는 7절에서 다섯 설정을 함께 놓고 확인합니다. Dropout은 검증 손실이 오르는 속도를 늦췄지만 오르는 것을 막지는 못했습니다.

## 5. Weight decay 더하기

**배경**: `torch.optim.AdamW`의 `weight_decay`는 갱신할 때마다 가중치를 조금씩 줄이므로, 0.5와 2.0 두 세기로 학습해 가중치 크기와 검증 손실이 세기에 따라 어떻게 달라지는지 봅니다.

**요구사항**
- (1) 세기 0.5, 2.0마다 Dropout 비율 0.0인 `make_rps_cnn` 모델 **`decay_model`** 을 새로 만듭니다.
- (2) 그 모델의 파라미터를 학습률 0.001, 그 세기의 `weight_decay`로 갱신하는 `torch.optim.AdamW` 옵티마이저 **`decay_optimizer`** 를 만듭니다. (교안 02 4-1에서 본 대로 `torch.optim.Adam`의 `weight_decay`는 AdamW와 다른 방식입니다.)
- (3) `fit_rps`로 50에폭 학습한 기록 표를 `decay_histories`(세기 -> 기록 표, 골격에 있음)에 담습니다.
- (4) `weight decay`, `가중치 크기`, `최저 검증 손실`, `마지막 검증 손실` 네 키의 dict를 모은 표 **`decay_table`** 을 만듭니다. 첫 행은 골격에 적힌 규제 없음(세기 0.0) 행입니다. 가중치 크기는 준비 셀의 `weight_size`(교안 02, 절편 제외)로 구합니다.

**예시**: 세기 0.0은 규제 없음이므로 `decay_table`의 첫 행이 `base_model`의 가중치 크기와 `base_history`의 검증 손실을 담습니다.

**확인 기준**: `decay_table`은 세기 0.0, 0.5, 2.0 순서의 3행이고, 세기가 커질수록 가중치 크기와 마지막 검증 손실이 작아집니다. 두 세기의 모델(`decay_models`)은 Dropout 비율이 0.0이고 옵티마이저(`decay_optimizers`)는 `torch.optim.AdamW`로 `weight_decay`가 그 세기입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 4-2의 weight_decay 반복문과 같은 모양이고, 한 행은 골격에 적힌 첫 행과 같은 네 키입니다.

세부구현:
1. torch.optim.AdamW는 Adam과 같은 인자에 weight_decay 하나가 더 있습니다. 반복 변수 weight_decay를 넘깁니다.
2. 모델은 Dropout이 섞이지 않게 비율 0.0으로 만듭니다.
3. 최저 검증 손실은 기록 표 검증 손실 열의 min, 마지막 검증 손실은 iloc에 -1을 준 값입니다.
4. 가중치 크기는 학습을 마친 decay_model로 구합니다.
```

</details>

In [ ]:
# [작성] weight decay 0.5, 2.0으로 학습해 규제 없음과 비교하세요.
decay_histories = {}
decay_models = {}
decay_optimizers = {}
decay_rows = [{"weight decay": 0.0, "가중치 크기": weight_size(base_model),
               "최저 검증 손실": base_history["검증 손실"].min(), "마지막 검증 손실": base_history["검증 손실"].iloc[-1]}]
for weight_decay in [0.5, 2.0]:
    # 1. 새 모델과 AdamW
    decay_model = ...
    decay_optimizer = ...
    decay_models[weight_decay] = decay_model  # 제공: 자가채점이 확인할 수 있게 보관
    decay_optimizers[weight_decay] = decay_optimizer

    # 2. 50에폭 학습
    history = ...
    decay_histories[weight_decay] = history

    # 3. 가중치 크기와 검증 손실로 한 행
    decay_rows.append(...)

decay_table = pd.DataFrame(decay_rows)
display(decay_table.round(3))

In [ ]:
# [자가채점] 옵티마이저 종류, 모델 구성, 세기에 따라 가중치 크기와 마지막 검증 손실이 줄어드는지 확인합니다.
assert sorted(decay_models) == [0.5, 2.0] and sorted(decay_optimizers) == [0.5, 2.0], "세기 0.5, 2.0마다 모델과 옵티마이저를 만들어 제공 줄로 보관하세요. `...`를 채웠는지 확인하세요."
for strength in [0.5, 2.0]:
    assert decay_models[strength][7].p == 0.0, f"세기 {strength}의 모델에 Dropout이 섞여 있습니다. weight decay만의 효과를 보려면 Dropout 비율은 0.0입니다."
    assert type(decay_optimizers[strength]) is torch.optim.AdamW, f"세기 {strength}의 옵티마이저가 {type(decay_optimizers[strength]).__name__}입니다. weight decay는 torch.optim.AdamW로 줍니다."
    assert decay_optimizers[strength].param_groups[0]["weight_decay"] == strength and decay_optimizers[strength].param_groups[0]["lr"] == 0.001, f"세기 {strength}의 옵티마이저는 학습률 0.001, weight_decay {strength}입니다."
    assert decay_optimizers[strength].param_groups[0]["params"][0] is decay_models[strength][0].weight, f"세기 {strength}의 옵티마이저는 그 세기의 모델 파라미터로 만드세요."

assert isinstance(decay_table, pd.DataFrame) and len(decay_table) == 3, "decay_table은 세기 0.0, 0.5, 2.0의 3행입니다."
assert list(decay_table.columns) == ["weight decay", "가중치 크기", "최저 검증 손실", "마지막 검증 손실"], "decay_table의 열은 weight decay, 가중치 크기, 최저 검증 손실, 마지막 검증 손실입니다."
assert list(decay_table["weight decay"]) == [0.0, 0.5, 2.0], "행은 세기 0.0, 0.5, 2.0 순서입니다."
assert sorted(decay_histories) == [0.5, 2.0] and len(decay_histories[2.0]) == 50, "decay_histories에는 세기 0.5, 2.0의 50에폭 기록 표가 들어갑니다."
assert abs(decay_table["마지막 검증 손실"].iloc[2] - decay_histories[2.0]["검증 손실"].iloc[-1]) < 1e-9, "마지막 검증 손실은 그 세기 기록 표의 마지막 행 값입니다."

sizes = list(decay_table["가중치 크기"])
assert sizes[0] > sizes[1] > sizes[2], f"가중치 크기가 {[round(s, 2) for s in sizes]}입니다. 세기가 커질수록 작아져야 합니다. 세기마다 새 모델을 만들었는지, 학습한 모델로 크기를 구했는지 확인하세요."
for strength in [0.5, 2.0]:
    assert abs(decay_table["가중치 크기"].iloc[[0.5, 2.0].index(strength) + 1] - weight_size(decay_models[strength])) < 1e-6, f"세기 {strength}의 가중치 크기는 학습을 마친 그 세기의 모델로 구합니다."
lasts = list(decay_table["마지막 검증 손실"])
assert lasts[0] > lasts[2], f"세기 2.0의 마지막 검증 손실({lasts[2]:.3f})이 규제 없음({lasts[0]:.3f})보다 작지 않습니다. 세기마다 새 모델을 만들었는지 확인하세요."
print("✅ 5번 통과")

**결과 읽기**
- **가중치 크기 6.793, 6.079, 4.887:** 세기 0.0, 0.5, 2.0에서 줄었고, 50에폭의 검증 손실도 1.805, 1.435, 1.105로 낮아졌습니다.
- **최저 검증 손실 0.937, 0.917, 0.910:** 거의 같습니다. 이 데이터에서 weight decay는 과적합이 진행되는 속도를 늦출 뿐, 처음 보는 손을 가장 잘 맞히는 수준을 높이지는 못했습니다. 학습 그림에 손 모델이 5개뿐이라는 문제는 가중치를 작게 해서는 풀리지 않습니다.

## 6. 데이터 증강 더하기

학습 그림이 부족하면 가진 그림을 조금씩 바꿔 매 에폭 다른 그림처럼 보여 줄 수 있습니다. 손 그림은 좌우로 뒤집어도 같은 손 모양(왼손으로 낸 가위도 가위)이고, 손이 1~2픽셀 옮겨져도 손 모양은 그대로입니다.

### 6-1. 증강 함수 만들기

**배경**: 교안 02 5-1의 증강에 쓴 `torch.roll`은 밀려난 행과 열을 반대쪽 가장자리로 다시 넣는데, 손글씨 숫자는 가장자리 값이 0(바탕, 교안 그림은 `gray_r`라 흰색으로 보임)이라 문제가 없지만 이 손 그림은 바탕 값이 1.0(흰색)이고 손목이 아래쪽 가장자리에 닿아 있습니다. `torch.roll`로 위로 옮기면 손목 조각이 아래로 밀려 나가는 대신 반대쪽(위쪽) 가장자리에 나타나므로, 사방을 흰 바탕으로 채운 그림에서 같은 크기를 잘라 옮깁니다. 여기에 정답이 바뀌지 않는 좌우 뒤집기를 그림마다 절반 확률로 더합니다.

**요구사항**
- (1) **`augment_rps(X_batch)`** 는 미니배치 `X_batch`(N, 1, 48, 48)를 받아 그림마다 따로 바꾼 사본(같은 모양)을 돌려줍니다. 정답은 바꾸지 않습니다.
- (2) 좌우 뒤집기: 그림마다 `torch.randint`로 0 또는 1을 뽑은 **`flip`** 이 1이면 그림을 좌우로 뒤집습니다. `torch.flip`은 `dims`로 준 축의 순서를 거꾸로 하는 함수이고, 그림 한 장 (1, 48, 48)에서 가로는 2번 축입니다.
- (3) 옮기기: 골격이 뽑은 `dy`, `dx`(-2~2)만큼 그림을 옮기되, 밀려 나가 비는 자리는 흰 바탕(1.0)입니다. 골격이 만든 `padded`(사방 2칸을 1.0으로 채운 (1, 52, 52) 그림)에서 48x48 창을 잘라 `augmented[i]`에 넣습니다.

**예시**: `dy`가 2이면 그림이 2칸 아래로 옮겨지고 맨 위 2행은 흰 바탕이며, 맨 아래 2행은 사라집니다.

**확인 기준**: 모양이 그대로이고 받은 미니배치를 바꾸지 않습니다. 증강한 그림마다 원래 그림(또는 좌우로 뒤집은 그림)을 세로 -2~2칸, 가로 -2~2칸 옮기고 빈 자리를 1.0으로 채운 그림 가운데 하나와 같습니다(위아래로 뒤집거나 반대쪽으로 감기면 걸립니다). 여러 번 부르면 그림의 절반 정도가 좌우로 뒤집히고, 한 미니배치 안에서도 그림마다 뒤집힘과 옮긴 칸 수가 다릅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 5-1의 augment 반복문에서, 옮기기 전에 뒤집기 한 단계를 더하고 옮기기를 roll 대신 잘라내기로 바꾼 모양입니다.

세부구현:
1. randint는 끝값을 빼고 고르므로 범위를 0부터 2까지로 주면 0 또는 1이 나옵니다. item으로 숫자를 꺼냅니다.
2. flip의 dims는 리스트로 줍니다. 1번 축을 주면 위아래로 뒤집히는데, 손이 거꾸로 선 그림은 테스트에 없습니다.
3. padded에서 원래 그림은 세로와 가로 모두 2번 칸부터 시작합니다. 그림을 아래로 dy칸 옮기는 것은 자를 창의 시작 행을 dy칸 위로 옮기는 것과 같고, 가로도 같습니다. 창의 크기는 48입니다.
4. 뒤집은(또는 그대로인) image를 padded에 넣어야 뒤집기와 옮기기가 함께 적용됩니다.
```

</details>

In [ ]:
# [작성] 그림마다 무작위로 좌우 뒤집고 옮기는 증강 함수를 완성하세요.
def augment_rps(X_batch):
    """그림마다 절반 확률로 좌우로 뒤집고, 위아래와 좌우로 -2~2픽셀을 무작위로 골라 옮긴 사본을 돌려줍니다. 빈 자리는 흰 바탕입니다."""
    augmented = torch.empty_like(X_batch)
    for i in range(len(X_batch)):
        image = X_batch[i]

        # 1. 0 또는 1을 뽑아 1이면 좌우로 뒤집기(그림 한 장 (1, 48, 48)에서 가로는 2번 축, dims는 리스트)
        flip = ...
        if flip == 1:
            image = ...

        # 2. 옮길 칸 수를 -2, -1, 0, 1, 2 가운데서 고르기(randint는 끝값 3을 빼고 고릅니다)
        dy = torch.randint(-2, 3, (1,)).item()
        dx = torch.randint(-2, 3, (1,)).item()

        # 3. 사방 2칸을 흰 바탕(1.0)으로 채운 (1, 52, 52) 그림(제공)에서 48 x 48 창을 잘라 옮기기
        padded = nn.functional.pad(image, (2, 2, 2, 2), value=1.0)
        augmented[i] = ...
    return augmented

In [ ]:
# [자가채점] 증강 결과의 모양, 옮긴 뒤 빈 자리, 뒤집는 방향, 이동 범위, 그림마다 따로 바꾸는지 확인합니다.
check_batch = X_valid[:16].clone()


def same_as_shifted(out, base, dy, dx):
    """out이 base를 아래로 dy칸, 오른쪽으로 dx칸 옮기고 빈 자리를 1.0으로 채운 그림과 같으면 True입니다."""
    expected = torch.ones_like(base)
    expected[:, max(dy, 0):48 + min(dy, 0), max(dx, 0):48 + min(dx, 0)] = base[:, max(-dy, 0):48 + min(-dy, 0), max(-dx, 0):48 + min(-dx, 0)]
    return torch.allclose(out, expected, atol=1e-5)


torch.manual_seed(0)
flipped_count = 0
moved_count = 0
mixed_trials = 0  # 한 미니배치 안에 뒤집힌 그림과 그대로인 그림이 섞인 횟수
varied_trials = 0  # 한 미니배치 안에 옮긴 칸 수가 다른 그림이 있는 횟수
for trial in range(20):
    augmented = augment_rps(check_batch)
    assert isinstance(augmented, torch.Tensor) and augmented.shape == check_batch.shape, "augment_rps는 받은 미니배치와 같은 (N, 1, 48, 48) 모양의 텐서를 돌려줍니다. `...`를 채웠는지 확인하세요."
    assert torch.equal(check_batch, X_valid[:16]), "augment_rps가 받은 미니배치 자체를 바꿨습니다. 바꾼 그림은 augmented에 넣으세요."
    trial_flips = 0
    trial_shifts = []
    for i in range(len(check_batch)):
        found = []
        for flipped in [False, True]:
            base = torch.flip(check_batch[i], dims=[2]) if flipped else check_batch[i]
            for dy in range(-2, 3):
                for dx in range(-2, 3):
                    if same_as_shifted(augmented[i], base, dy, dx):
                        found.append((flipped, dy, dx))
        assert len(found) > 0, "증강한 그림이 원래 그림을 좌우로 뒤집고 위아래, 좌우로 -2~2칸 옮긴 그림이 아닙니다. 위아래로 뒤집었거나 3칸 넘게 옮겼거나, 밀려 나간 부분이 반대쪽에 다시 나타났거나, 비는 자리가 흰 바탕(1.0)이 아닌지 확인하세요."
        flips = {f for f, _, _ in found}
        if flips == {True}:
            flipped_count += 1
            trial_flips += 1
        shift = (found[0][1], found[0][2])
        if shift != (0, 0):
            moved_count += 1
        trial_shifts.append(shift)
    if 0 < trial_flips < len(check_batch):
        mixed_trials += 1
    if len(set(trial_shifts)) > 1:
        varied_trials += 1

n_checked = 20 * len(check_batch)
assert 0.3 * n_checked < flipped_count < 0.7 * n_checked, f"그림 {n_checked}장 가운데 {flipped_count}장이 좌우로 뒤집혔습니다. 그림마다 0 또는 1을 뽑아 1일 때만 뒤집어야 합니다."
assert mixed_trials >= 15, f"미니배치 20번 가운데 {mixed_trials}번만 뒤집힌 그림과 그대로인 그림이 섞였습니다. 미니배치 전체를 한 번에 뒤집지 말고, 반복문 안에서 그림마다 따로 뽑아 뒤집으세요."
assert moved_count > 0.6 * n_checked, f"그림 {n_checked}장 가운데 {moved_count}장만 옮겨졌습니다. 골격의 dy, dx만큼 옮겼는지 확인하세요."
assert varied_trials >= 15, f"미니배치 20번 가운데 {varied_trials}번만 그림마다 옮긴 칸 수가 달랐습니다. 반복문 안에서 그림 한 장(image)을 옮겼는지 확인하세요."
print(f"✅ 6-1번 통과: 그림 {n_checked}장 가운데 뒤집힘 {flipped_count}장 | 옮겨짐 {moved_count}장")

아래 셀은 실행만 하세요. 학습 그림 6장(위)과 `augment_rps`를 한 번 거친 같은 그림(아래)을 비교합니다.

In [ ]:
# 원래 그림(위)과 증강한 그림(아래)
torch.manual_seed(1)
original = X_train[:6]
augmented = augment_rps(original)
fig, axes = plt.subplots(2, 6, figsize=(14, 6))
for k in range(6):
    axes[0, k].imshow(original[k, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, k].set_title(f"원래, {class_names[y_train[k]]}", fontsize=11)
    axes[1, k].imshow(augmented[k, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, k].set_title(f"증강, {class_names[y_train[k]]}", fontsize=11)
    axes[0, k].axis("off")
    axes[1, k].axis("off")
plt.suptitle("augment_rps를 거친 학습 그림(정답은 그대로)", fontsize=14)
plt.tight_layout()
plt.show()

**결과 읽기**
- **통과 메시지의 뒤집힘 154장:** 320장 가운데 절반 안팎이 좌우로 뒤집혔고, 311장이 옮겨졌습니다.
- **위아래 그림 비교:** 증강한 그림은 손이 조금 옮겨지거나 뒤집혔고, 옮겨져 빈 가장자리는 흰 바탕입니다. 손목이 위쪽 가장자리에 나타난 그림이 없습니다.

### 6-2. 증강하며 한 에폭 학습하는 함수 만들기

**배경**: 교안 02는 에폭마다 학습 그림 전체를 증강해 DataLoader를 다시 만들었지만, 여기서는 9절 Lightning의 `training_step`과 같은 모양이 되도록 미니배치마다 증강하는 학습 함수를 만듭니다. 검증, 테스트 그림은 증강하지 않으므로 `evaluate`는 그대로 씁니다.

**요구사항**
- (1) **`train_one_epoch_augment(model, loader, loss_fn, optimizer)`** 는 제공된 `train_one_epoch`과 같은 일을 하되 아래 두 가지가 다릅니다.
- (2) 맨 앞에서 모델을 학습 모드로 바꿉니다(Dropout이 있는 모델을 `evaluate` 뒤에 다시 학습하므로 꼭 필요합니다).
- (3) 미니배치를 장치로 옮긴 뒤 `augment_rps`로 그림만 바꿔 순전파합니다. 정답은 그대로 씁니다.

**예시**: 같은 미니배치를 두 번 넣어도 증강 때문에 모델이 받는 그림이 매번 다릅니다.

**확인 기준**: 평가 모드인 모델을 넣어도 순전파가 모두 학습 모드에서 일어나고 함수가 끝난 뒤에도 학습 모드입니다. 모델이 받은 그림이 원래 그림과 다르고(증강됨), 미니배치 손실의 평균(숫자)을 돌려줍니다.

<details><summary>힌트</summary>

```text
접근방법:
- 준비 셀의 train_one_epoch 본문을 옆에 두고, 다른 두 자리만 채웁니다.

세부구현:
1. 학습 모드는 train_one_epoch 첫 줄과 같은 메서드로 바꿉니다.
2. 증강한 결과를 X_batch에 다시 담으면 아래 순전파가 증강한 그림을 씁니다.
3. 손실은 loss_fn에 모델의 로짓과 정답을 넘겨 구합니다.
```

</details>

In [ ]:
# [작성] 미니배치마다 증강하며 한 에폭 학습하는 함수를 완성하세요.
def train_one_epoch_augment(model, loader, loss_fn, optimizer):
    """미니배치마다 그림을 증강해 파라미터를 갱신하고, 미니배치 손실의 평균을 돌려줍니다."""
    # 1. 학습 모드로 바꾸기
    ...
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 2. 그림만 증강(정답은 그대로)
        X_batch = ...

        # 3. 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = ...
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# [자가채점] 학습 모드, 증강 여부, 돌려주는 값을 확인합니다.
check_model = make_rps_cnn(0.5)
check_model.eval()  # 일부러 평가 모드로 둔 뒤 함수를 부릅니다

# 모델이 실제로 받은 그림과 그때의 모드를 모아 두는 기록 함수(forward hook)
seen_batches = []
seen_modes = []


def record_input(module, inputs, output):
    """모델에 들어온 미니배치를 복사하고, 순전파할 때 학습 모드였는지 기록합니다."""
    seen_batches.append(inputs[0].detach().cpu().clone())
    seen_modes.append(module.training)


hook = check_model.register_forward_hook(record_input)
check_loader = DataLoader(TensorDataset(X_train[:128], y_train[:128]), batch_size=64, shuffle=False)
torch.manual_seed(0)
check_loss = train_one_epoch_augment(check_model, check_loader, nn.CrossEntropyLoss(), torch.optim.Adam(check_model.parameters(), lr=0.001))
hook.remove()

assert check_model.training, "함수가 끝났는데 모델이 평가 모드입니다. 맨 앞에서 학습 모드로 바꾸세요. 빠뜨리면 evaluate 뒤의 학습에서 Dropout이 꺼진 채 학습합니다."
assert len(seen_modes) > 0 and all(seen_modes), "순전파할 때 모델이 평가 모드였습니다. 학습 모드로 바꾸는 줄은 반복문보다 먼저, 함수 맨 앞에 두세요. 함수 끝에서 바꾸면 그 에폭은 Dropout이 꺼진 채 학습합니다."
assert isinstance(check_loss, float) and 0 < check_loss < 5, "train_one_epoch_augment는 미니배치 손실의 평균(숫자)을 돌려줍니다."
assert len(seen_batches) == 2, "미니배치 2개(128장, 배치 64)를 한 번씩 순전파해야 합니다."
same_count = 0
for k in range(2):
    if torch.equal(seen_batches[k], X_train[k * 64:(k + 1) * 64]):
        same_count += 1
assert same_count < 2, "모델이 받은 그림이 원래 그림과 같습니다. 순전파 전에 augment_rps로 미니배치 그림을 증강하세요."
print("✅ 6-2번 통과: 미니배치 손실의 평균", round(check_loss, 3))

**결과 읽기**
- **미니배치 2개:** 128장을 64장씩 두 번 순전파했고, 모델이 받은 그림이 원래 그림과 다릅니다(증강됨).
- **함수가 끝난 뒤 학습 모드:** 평가 모드로 넣은 모델도 이 함수 안에서는 학습 모드로 바뀌었습니다.

### 6-3. 증강, 증강 + Dropout으로 50에폭 학습하기

**배경**: 증강만의 효과와, 증강에 Dropout을 더한 효과를 함께 보려고 Dropout 비율만 다른 두 모델을 증강하며 학습합니다.

**요구사항**
- (1) Dropout 비율 0.0인 `make_rps_cnn` 모델 **`aug_model`** 을 만듭니다.
- (2) Dropout 비율 0.5인 모델 **`combo_model`** 을 만듭니다.
- (3) 모델마다 학습률 0.001인 Adam 옵티마이저 **`aug_optimizer`**, **`combo_optimizer`** 를 만듭니다.
- (4) `fit_rps`에 증강을 켜고 50에폭 학습한 표 **`aug_history`**, **`combo_history`**(각 50행)를 만듭니다.

**예시**: 3-2와 같은 방법이고, `fit_rps`의 마지막 인자 `use_augment`만 True입니다.

**확인 기준**: 두 표 모두 50행이고 `aug_model`의 Dropout 비율은 0.0, `combo_model`은 0.5입니다. 두 설정 모두 최저 검증 손실이 규제 없음보다 0.2 넘게 낮습니다. 검증 데이터 `X_valid`는 증강하지 않아 원래 그림 그대로입니다.

<details><summary>힌트</summary>

```text
세부구현:
1. 3-2와 같은 방법이고, fit_rps의 마지막 인자 use_augment만 True로 줍니다.
```

</details>

In [ ]:
# [작성] 증강과, 증강 + Dropout 0.5로 각각 50에폭 학습하세요.

# 1. 증강만
aug_model = ...
aug_optimizer = ...
aug_history = ...
display(aug_history.set_index("에폭").tail().round(3))

# 2. 증강 + Dropout 0.5
combo_model = ...
combo_optimizer = ...
combo_history = ...
display(combo_history.set_index("에폭").tail().round(3))

In [ ]:
# [자가채점] 두 설정의 학습 결과와 검증 데이터를 확인합니다.
assert isinstance(aug_history, pd.DataFrame) and len(aug_history) == 50, "aug_history는 50에폭의 표(50행)입니다. `...`를 채웠는지 확인하세요."
assert isinstance(combo_history, pd.DataFrame) and len(combo_history) == 50, "combo_history는 50에폭의 표(50행)입니다."
assert aug_model[7].p == 0.0, "aug_model은 증강 효과만 보려고 Dropout 비율 0.0으로 만듭니다."
assert combo_model[7].p == 0.5, "combo_model은 Dropout 비율 0.5로 만듭니다."
assert aug_optimizer.param_groups[0]["params"][0] is aug_model[0].weight and combo_optimizer.param_groups[0]["params"][0] is combo_model[0].weight, "각 옵티마이저는 그 모델의 파라미터로 만드세요."
assert torch.equal(X_valid, X_all[valid_index]), "X_valid가 원래 그림과 다릅니다. 증강은 학습 미니배치에만 합니다. 검증 데이터를 증강했다면 1절부터 다시 실행하세요."
base_best = base_history["검증 손실"].min()
aug_best = aug_history["검증 손실"].min()
combo_best = combo_history["검증 손실"].min()
assert aug_best < base_best - 0.2 and combo_best < base_best - 0.2, f"최저 검증 손실(증강 {aug_best:.3f}, 증강 + Dropout {combo_best:.3f})이 규제 없음({base_best:.3f})보다 0.2 넘게 낮지 않습니다. fit_rps에서 증강을 켰는지 확인하세요."
aug_best_accuracy = aug_history.loc[aug_history["검증 손실"].idxmin(), "검증 정확도"]
print("✅ 6-3번 통과: 최저 검증 손실(증강)", round(aug_best, 3), "| 그 에폭의 검증 정확도", round(aug_best_accuracy, 3))

**결과 읽기**
- **최저 검증 손실 0.346:** 규제 없음(0.937)보다 크게 낮아졌습니다. 증강 모델의 그 에폭(31에폭) 검증 정확도는 0.867입니다.
- **증강 + Dropout 0.281:** 증강만(0.346)과의 차이는 규제 없음과 증강의 차이(0.937과 0.346)보다 훨씬 작습니다. 학습 그림 600장을 에폭마다 다르게 뒤집고 옮겨 보여 주니, 손의 위치나 방향보다 손가락 모양을 보고 맞히도록 배웠습니다.

### 6-4. 손 그림에 쓸 증강 고르기

**배경**: 6-1은 직접 만들기 쉬운 좌우 뒤집기와 옮기기만 썼습니다. 교안 02 5-1의 증강은 정답이 바뀌지 않으면서 실제 입력에 있을 법한 그림을 만드는 변환만 쓰므로, 학습 그림의 가장자리를 숫자로 확인하고 아래 후보 여섯 가지 가운데 손 그림에 쓸 증강을 고릅니다.

| 후보 | 하는 일 |
|---|---|
| `좌우 뒤집기` | 좌우를 뒤집습니다 |
| `위아래 뒤집기` | 위아래를 뒤집습니다 |
| `작은 각도 회전` | -15°~15° 사이 각도로 돌립니다 |
| `180도 회전` | 위아래와 좌우를 함께 뒤집어 거꾸로 세웁니다 |
| `이동` | 위아래, 좌우로 2픽셀 안에서 옮기고 빈 자리는 흰 바탕으로 채웁니다 |
| `흑백 반전` | 픽셀 값 x를 1 - x로 바꿉니다. 흰 바탕은 검게, 어두운 손은 밝게 됩니다 |

**요구사항**
- (1) 학습 그림 `X_train`(600장) 가운데, 값이 0.8보다 작은 픽셀(손)이 맨 아래 행에 하나라도 있는 그림의 비율을 파이썬 실수 **`bottom_touch_rate`** 로 만듭니다.
- (2) 같은 방법으로 맨 위 행의 비율 **`top_touch_rate`** 를 만듭니다.
- (3) `augment_candidates`(위 표의 후보 이름, 제공 줄)에서 손 그림에 쓸 증강만 표 순서대로 고른 리스트 **`chosen_augments`** 를 만듭니다.

**예시**: 손글씨 숫자에는 좌우 뒤집기를 쓰지 않습니다. 2, 7처럼 거울에 비친 다른 모양이 되기 때문입니다(교안 02 5-1).

**확인 기준**: `bottom_touch_rate`, `top_touch_rate`가 학습 그림 600장 기준 비율(0~1 실수)입니다. `chosen_augments`는 `augment_candidates`의 이름만 표 순서대로 담고, 정답이 바뀌거나 실제 입력에 없는 그림을 만드는 후보를 넣지 않으며, 쓸 수 있는 후보를 빠뜨리지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 행 하나를 골라 그림마다 손 픽셀이 있는지 True/False로 만든 뒤 평균을 냅니다. 후보마다 두 기준(정답이 그대로인가, 실제 입력에 있을 법한가)을 차례로 묻습니다.

세부구현:
1. X_train의 모양은 (600, 1, 48, 48)이고 축 번호는 0부터 셉니다. 2번 축이 세로(행), 3번 축이 가로(열)이므로, 맨 아래 행은 2번 축(세로)의 마지막 번호(-1), 맨 위 행은 2번 축의 0번입니다.
2. 비교 연산으로 만든 True/False 텐서에 any와 dim을 주면, 그 축에 True가 하나라도 있는지를 그림마다 돌려줍니다.
3. True/False를 float로 바꿔 mean을 구하면 True의 비율입니다.
4. 위아래로 뒤집거나 거꾸로 세운 그림에서 손목이 어느 가장자리에 닿을지, 두 비율과 아래 그림으로 판단합니다. 준비 절에서 본 것처럼 같은 손 모델의 120장은 손을 조금씩 돌려 그린 그림입니다.
```

</details>

아래 셀은 실행만 하세요. 학습 그림 한 장에 후보 네 가지를 적용한 모습입니다(작은 각도 회전과 이동은 준비 절의 이웃한 그림과 6-1의 그림에서 봤습니다).

In [ ]:
# 학습 그림 한 장 (48, 48)에 후보 변환 네 가지를 적용합니다. 0번 축이 세로, 1번 축이 가로입니다
sample_image = X_train[0, 0]
candidate_images = {
    "원래 그림": sample_image,
    "좌우 뒤집기": torch.flip(sample_image, dims=[1]),
    "위아래 뒤집기": torch.flip(sample_image, dims=[0]),
    "180도 회전": torch.flip(sample_image, dims=[0, 1]),
    "흑백 반전": 1 - sample_image,
}
fig, axes = plt.subplots(1, 5, figsize=(14, 3.4))
for k, (name, image) in enumerate(candidate_images.items()):
    axes[k].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[k].set_title(name, fontsize=12)
    axes[k].axis("off")
plt.suptitle(f"후보 변환을 적용한 학습 그림(정답 {class_names[y_train[0]]})", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# [작성] 학습 그림의 맨 아래 행과 맨 위 행을 확인하고, 손 그림에 쓸 증강을 고르세요.

# 1. 손 픽셀(값 0.8 미만)이 맨 아래 행, 맨 위 행에 있는 그림의 비율
bottom_touch_rate = ...
top_touch_rate = ...
print("맨 아래 행에 손이 닿은 그림:", round(bottom_touch_rate, 3), "| 맨 위 행:", round(top_touch_rate, 3))

# 2. 후보(제공)에서 손 그림에 쓸 증강만 표 순서대로 고르기
augment_candidates = ["좌우 뒤집기", "위아래 뒤집기", "작은 각도 회전", "180도 회전", "이동", "흑백 반전"]
chosen_augments = ...
print("고른 증강:", chosen_augments)

In [ ]:
# [자가채점] 두 비율과 고른 증강을 확인합니다.
import hashlib

assert isinstance(bottom_touch_rate, float) and isinstance(top_touch_rate, float), "bottom_touch_rate, top_touch_rate는 파이썬 실수입니다. 비율 텐서에서 숫자를 꺼냈는지, `...`를 채웠는지 확인하세요."
for name, rate in [("bottom_touch_rate", bottom_touch_rate), ("top_touch_rate", top_touch_rate)]:
    assert 0.0 <= rate <= 1.0 and abs(rate * 600 - round(rate * 600)) < 1e-4, f"{name}({rate:.4f})가 학습 그림 600장 기준 비율이 아닙니다. 그림마다 True/False 하나를 만든 뒤 X_train 600장의 평균을 구하세요."
assert bottom_touch_rate > 0.95, f"bottom_touch_rate가 {bottom_touch_rate:.3f}입니다. 손목은 거의 모든 그림에서 아래쪽 가장자리에 닿습니다. 맨 아래 행(2번 축의 마지막 번호)을 골랐는지, 0.8보다 작은 픽셀을 손으로 봤는지 확인하세요."
assert top_touch_rate < 0.1, f"top_touch_rate가 {top_touch_rate:.3f}입니다. 맨 위 행(2번 축의 0번)을 골랐는지 확인하세요."

# 고른 증강: 후보 이름만, 표 순서대로, 이름마다 한 번씩
assert isinstance(chosen_augments, list) and len(chosen_augments) > 0 and all(isinstance(name, str) for name in chosen_augments), "chosen_augments는 후보 이름(문자열)을 담은 리스트입니다."
assert all(name in augment_candidates for name in chosen_augments), "chosen_augments에는 augment_candidates에 있는 이름만 그대로 적습니다."
assert chosen_augments == sorted(chosen_augments, key=augment_candidates.index) and len(set(chosen_augments)) == len(chosen_augments), "chosen_augments는 표 순서대로, 이름마다 한 번씩 적습니다."

# 고른 조합을 해시 지문으로 비교합니다(정답 이름은 셀에 적지 않습니다)
fingerprint = hashlib.sha256("|".join(sorted(chosen_augments)).encode()).hexdigest()
assert fingerprint == "648da8c999bb486e89468a5400cb68e1dc2c09b5321ad7d9d8d8b67ae65eb73e", "고른 증강의 조합이 맞지 않습니다. 후보마다 두 기준(정답이 그대로인가, 실제 입력에 있을 법한가)을 다시 판단하세요. 손목이 닿는 가장자리(두 비율과 위 그림), 학습, 테스트 그림의 바탕 색(6-1 배경), 같은 손 모델 120장을 그린 방법(준비 절), 6-1에서 쓴 변환을 근거로 씁니다."
print("✅ 6-4번 통과")

**결과 읽기**
- **맨 아래 행 1.0, 맨 위 행 0.058:** 학습 그림 600장 모두 손이 아래쪽 가장자리에 닿아 있고, 위쪽 가장자리에 닿은 그림은 0.058뿐입니다. 위아래로 뒤집거나 거꾸로 세운 그림은 손목이 위쪽에 닿아 실제 입력에 없는 그림이므로, 정답(손 모양)이 그대로여도 쓰지 않습니다.
- **좌우 뒤집기, 작은 각도 회전, 이동:** 정답이 그대로이고 실제 입력에도 있는 변화(왼손, 손 각도, 손 위치)입니다. 흑백 반전은 바탕이 검게 바뀌어 흰 바탕뿐인 입력과 다릅니다.

## 7. 검증 손실로 설정 비교하고 고르기

**배경**: 교안 02 7-1의 하이퍼파라미터 탐색처럼 설정은 검증 손실로 고르고, 테스트 데이터는 고른 설정을 마지막에 한 번 평가할 때만 씁니다(테스트 정확도로 고르면 그 정확도는 처음 보는 데이터의 성능보다 좋게 나옵니다). 교안 02가 정한 규칙은 최저 검증 손실이 가장 낮은 설정을 고르는 것입니다. 마지막 에폭의 검증 손실로 고르지 않습니다.

**요구사항**
- (1) `all_histories`(설정 이름 -> 기록 표, 골격에 있음)를 돌며 설정마다 `설정`, `최저 검증 손실`, `그 에폭`, `마지막 검증 손실` 네 키의 dict를 모은 표 **`compare_table`**(5행, `all_histories` 순서)을 만듭니다.
- (2) `그 에폭`은 검증 손실이 가장 낮았던 에폭이고, 정수(`int`)로 담습니다.
- (3) 위 열 이름의 표를 받아 최저 검증 손실이 가장 작은 행의 `설정` 값을 돌려주는 함수 **`choose_setting(table)`** 을 만듭니다.
- (4) `choose_setting(compare_table)`의 결과 **`best_setting`** 을 만듭니다.

**예시**: 표가 (가: 최저 0.5, 마지막 0.6), (나: 최저 0.3, 마지막 0.9), (다: 최저 0.4, 마지막 0.35)이면 마지막 검증 손실이 가장 낮은 설정은 다이지만, 규칙에 맞는 설정은 최저 검증 손실이 가장 낮은 나입니다.

**확인 기준**: `compare_table`은 `all_histories` 순서의 5행이고 값이 각 기록 표와 맞습니다. `choose_setting`은 최저 검증 손실이 가장 작은 설정을 고르고(마지막 검증 손실이 가장 낮은 설정을 고르면 걸립니다), `best_setting`은 `compare_table`에서 그 규칙으로 고른 설정입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 설정마다 3-3과 같은 방법으로 가장 낮은 행을 찾아 다섯 행을 모으고, 그 표에서 다시 한 번 가장 작은 행을 찾는 일을 함수로 만듭니다.

세부구현:
1. best_row는 3-3의 base_best_row와 같은 방법으로 찾습니다. 에폭은 int로 바꿉니다.
2. 마지막 검증 손실은 기록 표 검증 손실 열의 iloc에 -1을 준 값입니다.
3. choose_setting은 표의 최저 검증 손실 열에서 idxmin으로 행 이름을 찾고, 그 행의 설정 칸을 읽습니다.
```

</details>

In [ ]:
# [작성] 다섯 설정을 검증 손실로 비교해 하나를 고르세요.
all_histories = {
    "규제 없음": base_history,
    "Dropout 0.5": drop_history,
    "weight decay 2.0": decay_histories[2.0],
    "증강": aug_history,
    "증강 + Dropout 0.5": combo_history,
}


def choose_setting(table):
    """최저 검증 손실이 가장 작은 행의 설정 이름을 돌려줍니다."""
    ...


compare_rows = []
for setting, history in all_histories.items():
    # 1. 검증 손실이 가장 낮은 행
    best_row = ...

    # 2. 설정마다 한 행
    compare_rows.append(...)

compare_table = pd.DataFrame(compare_rows)
display(compare_table.round(3))

# 3. 최저 검증 손실이 가장 작은 설정
best_setting = ...
print("고른 설정:", best_setting)

In [ ]:
# [자가채점] 비교 표, 고르는 규칙, 고른 설정을 확인합니다.
assert isinstance(compare_table, pd.DataFrame) and len(compare_table) == 5, "compare_table은 설정마다 한 행, 5행입니다. `...`를 채웠는지 확인하세요."
assert list(compare_table.columns) == ["설정", "최저 검증 손실", "그 에폭", "마지막 검증 손실"], "compare_table의 열은 설정, 최저 검증 손실, 그 에폭, 마지막 검증 손실입니다."
assert list(compare_table["설정"]) == list(all_histories), "행은 all_histories 순서입니다."
for k, history in enumerate(all_histories.values()):
    row = compare_table.iloc[k]
    assert abs(row["최저 검증 손실"] - history["검증 손실"].min()) < 1e-9, f"{row['설정']} 행의 최저 검증 손실이 기록 표와 다릅니다. 검증 손실 열의 가장 작은 값을 쓰세요."
    epoch_loss = history.loc[history["에폭"] == row["그 에폭"], "검증 손실"]
    assert len(epoch_loss) == 1 and epoch_loss.iloc[0] == history["검증 손실"].min(), f"{row['설정']} 행의 그 에폭이 맞지 않습니다. 행 번호가 아니라 에폭 열의 값을 쓰세요."
    assert abs(row["마지막 검증 손실"] - history["검증 손실"].iloc[-1]) < 1e-9, f"{row['설정']} 행의 마지막 검증 손실은 기록 표 마지막 행의 값입니다."
assert compare_table["그 에폭"].dtype.kind == "i", "그 에폭 열은 정수로 바꾼 값입니다. 꺼낸 행의 에폭은 10.0처럼 실수로 나옵니다."

# 고르는 규칙: 최저 검증 손실이 가장 작은 설정. 두 열의 가장 작은 값이 서로 다른 설정에 있는 작은 표로 확인합니다
small_table = pd.DataFrame({"설정": ["가", "나", "다"], "최저 검증 손실": [0.5, 0.3, 0.4], "그 에폭": [3, 5, 4], "마지막 검증 손실": [0.6, 0.9, 0.35]})
assert choose_setting(small_table) == "나", f"choose_setting이 {choose_setting(small_table)}을 골랐습니다. 최저 검증 손실이 가장 작은 설정(나)을 골라야 합니다. 마지막 검증 손실이 아니라 최저 검증 손실 열을 보세요."
assert best_setting in list(compare_table["설정"]), "best_setting은 compare_table 설정 열의 값 하나(설정 이름)입니다."
assert best_setting == choose_setting(compare_table), "best_setting은 choose_setting(compare_table)의 결과입니다."
chosen_loss = compare_table[compare_table["설정"] == best_setting]["최저 검증 손실"].iloc[0]
assert chosen_loss == compare_table["최저 검증 손실"].min(), f"best_setting이 {best_setting}입니다. 최저 검증 손실 열에서 값이 가장 작은 행의 설정을 고르세요."
print("✅ 7번 통과")

아래 두 셀은 실행만 하세요. 다섯 설정의 에폭별 검증 손실을 그리고, 설정마다 교안 02 2-3의 **에폭 간 변화**(15에폭 뒤 구간에서 이웃한 두 에폭의 검증 손실 차이를 평균한 값)를 계산합니다. 최저 검증 손실의 차이가 이보다 작으면 설정의 차이로 단정하지 않습니다.

In [ ]:
# 다섯 설정의 검증 손실 곡선
fig, ax = plt.subplots(figsize=(14, 5))
for setting, history in all_histories.items():
    ax.plot(history["에폭"], history["검증 손실"], marker="o", markersize=3, label=setting)
ax.axhline(math.log(3), color="gray", linestyle="--", linewidth=1, label="log 3")
ax.set_title("설정별 검증 손실(학습 600장, 검증은 처음 보는 손 모델 2개)", fontsize=14)
ax.set_xlabel("에폭", fontsize=12)
ax.set_ylabel("검증 손실", fontsize=12)
ax.tick_params(labelsize=11)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=11)  # 범례가 곡선을 가리지 않게 그래프 오른쪽 밖에 둡니다
plt.tight_layout()
plt.show()

In [ ]:
# 설정마다 에폭 간 변화(교안 02 2-3)를 계산해 최저 검증 손실의 차이와 비교합니다
for setting, history in all_histories.items():
    print(setting, "| 에폭 간 변화:", round(epoch_change(history), 3))

**결과 읽기**
- **최저 검증 손실 0.937, 0.862, 0.910, 0.346, 0.281:** 규제 없음, Dropout, weight decay, 증강, 증강 + Dropout 순입니다. 증강을 쓴 두 설정만 0.4 넘게 낮아, 에폭 간 변화(0.035~0.066)보다 훨씬 큽니다. 규제 없음, Dropout, weight decay 사이의 차이(0.027~0.075)는 에폭 간 변화와 비슷하거나 작아 셋의 순서를 단정하지 않습니다.
- **곡선 모양:** 규제 없음 곡선은 10에폭 근처부터 오르고, Dropout 곡선은 0.85~1.4 사이에서 오르내리며, weight decay 곡선은 1.0 근처에서 거의 오르지도 내려가지도 않습니다. 증강을 쓴 두 곡선은 20에폭 근처까지 내려간 뒤 증강은 0.35~0.55, 증강 + Dropout은 0.28~0.5 사이에서 오르내립니다.
- **고른 설정 증강 + Dropout 0.5:** 정한 규칙대로 최저 검증 손실이 가장 낮은 설정입니다. 증강만과의 차이(0.346과 0.281)는 두 설정의 에폭 간 변화(0.065, 0.042)와 비슷한 크기라 두 설정은 비슷하게 좋은 후보입니다.

## 8. 조기 종료로 가장 좋은 모델 남기기

**배경**: 7절에서 고른 설정(증강 + Dropout 0.5)을 에폭 수를 정해 두지 않고, 교안 02 6절의 조기 종료(검증 손실이 `patience` 에폭 동안 낮아지지 않으면 멈추고 가장 낮았던 때의 파라미터로 되돌리기)로 학습합니다.

**요구사항**
- (1) 7절 결과와 관계없이 이 과제는 증강 + Dropout 0.5로 진행합니다. Dropout 비율 0.5인 `make_rps_cnn` 모델 **`es_model`** 을 만듭니다.
- (2) `es_model`의 파라미터를 학습률 0.001로 갱신하는 Adam 옵티마이저 **`es_optimizer`** 를 만듭니다. 최대 60에폭, `patience`는 10입니다(골격에 있음).
- (3) 에폭마다 `train_one_epoch_augment`로 학습하고 `evaluate`로 **검증 데이터**의 손실과 정확도를 구합니다.
- (4) 검증 손실이 지금까지 가장 낮은 값보다 낮으면 최저 검증 손실 **`best_valid_loss`** 와 그 에폭 **`best_epoch`** 를 바꿉니다.
- (5) 같은 때의 파라미터 복사본을 **`best_state`** 에 저장합니다.
- (6) 반복이 끝나면 `es_model`을 `best_state`의 파라미터로 되돌립니다. 기록 표는 **`es_history`** 입니다(골격에 있음).

**예시**: 10에폭이 가장 좋았고 그 뒤 10에폭(11~20) 동안 낮아지지 않으면 20에폭에서 멈추고, 10에폭의 파라미터로 되돌립니다.

**확인 기준**: 멈춘 에폭이 `best_epoch + patience`(또는 60)이고, `best_valid_loss`가 기록 표의 최저 검증 손실이면서 0.6보다 낮으며, 되돌린 `es_model`을 `valid_loader`로 다시 평가한 손실이 `best_valid_loss`와 거의 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 6-2의 fit_early_stopping 반복문을, 함수 없이 이 셀에 바로 쓴 모양입니다.

세부구현:
1. 학습은 train_one_epoch_augment, 평가는 evaluate에 valid_loader를 넘깁니다. 인자 순서는 3-1의 fit_rps와 같습니다.
2. state_dict는 모델이 지금 쓰는 텐서를 가리키기만 하므로, copy.deepcopy로 감싸 그때의 값을 복사해 둡니다.
3. 되돌리기는 fit_early_stopping의 마지막 줄과 같은 메서드(load_state_dict)입니다.
```

</details>

In [ ]:
# [작성] 증강 + Dropout 0.5를 조기 종료로 학습하세요.
import copy

# 1. 고른 설정의 새 모델과 옵티마이저
es_model = ...
es_optimizer = ...
loss_fn = nn.CrossEntropyLoss()
torch.manual_seed(0)  # 미니배치 순서, Dropout, 증강이 fit_rps와 같게 시드 고정

patience = 10
best_valid_loss = float("inf")  # 첫 에폭의 검증 손실이 무엇이든 더 작도록 무한대에서 시작
best_epoch = 0
best_state = None
es_rows = []
for epoch in range(1, 61):
    # 2. 한 에폭 학습(증강)하고 검증 데이터로 평가
    train_loss = ...
    valid_loss, valid_accuracy = ...
    es_rows.append({"에폭": epoch, "학습 손실": train_loss, "검증 손실": valid_loss, "검증 정확도": valid_accuracy})

    # 3. 지금까지의 최저 검증 손실보다 낮으면 그 값, 에폭, 파라미터 복사본을 저장
    if valid_loss < best_valid_loss:
        ...  # best_valid_loss, best_epoch, best_state 세 값을 바꾸기(세 줄)

    # 4. 가장 좋았던 에폭에서 patience 에폭이 지나도록 낮아지지 않으면 멈춤
    if epoch - best_epoch >= patience:
        print(f"{epoch}에폭에서 멈춤: {patience}에폭 동안 검증 손실이 나아지지 않았습니다")
        break

# 5. 검증 손실이 가장 낮았던 파라미터로 되돌리기
...  # es_model을 best_state의 파라미터로 되돌리기
es_history = pd.DataFrame(es_rows)
print("가장 좋았던 에폭:", best_epoch, "| 검증 손실:", round(best_valid_loss, 3))

In [ ]:
# [자가채점] 멈춘 시점, 가장 좋았던 에폭, 되돌린 파라미터를 확인합니다.
assert isinstance(es_history, pd.DataFrame) and len(es_history) > 0, "es_history는 에폭마다 한 행인 표입니다. `...`를 채웠는지 확인하세요."
assert es_model[7].p == 0.5, "es_model은 고른 설정(증강 + Dropout 0.5)이라 Dropout 비율이 0.5입니다."
assert es_optimizer.param_groups[0]["params"][0] is es_model[0].weight, "es_optimizer는 es_model의 파라미터로 만드세요."
assert abs(best_valid_loss - es_history["검증 손실"].min()) < 1e-9, "best_valid_loss는 기록 표의 검증 손실 가운데 가장 작은 값입니다. 검증 손실이 더 낮을 때 best_valid_loss를 바꿨는지 확인하세요."
assert es_history["검증 손실"].iloc[best_epoch - 1] == best_valid_loss, "best_epoch는 검증 손실이 가장 낮았던 에폭입니다."
stop_epoch = len(es_history)
assert stop_epoch == best_epoch + patience or stop_epoch == 60, f"{stop_epoch}에폭에서 멈췄는데 가장 좋았던 에폭은 {best_epoch}입니다. 검증 손실이 더 낮을 때 best_epoch를 그 에폭으로 바꿨는지 확인하세요."
assert best_valid_loss < 0.6, f"best_valid_loss가 {best_valid_loss:.3f}입니다. 증강 + Dropout 0.5이면 0.6보다 낮습니다. 반복문에서 증강하는 학습 함수를 썼는지 확인하세요."

recheck_loss, recheck_accuracy = evaluate(es_model, valid_loader, nn.CrossEntropyLoss())
assert abs(recheck_loss - best_valid_loss) < 0.005, f"되돌린 es_model의 검증 손실({recheck_loss:.4f})이 best_valid_loss({best_valid_loss:.4f})와 다릅니다. (1) 조기 종료 판단은 test_loader가 아니라 valid_loader의 손실로 합니다. (2) best_state에 복사본을 저장했는지, 마지막에 es_model을 best_state로 되돌렸는지 확인하세요."
print("✅ 8번 통과: 멈춘 에폭", stop_epoch, "| 가장 좋았던 에폭", best_epoch)

**결과 읽기**
- **45에폭에 가장 낮은 검증 손실 0.281:** 그 뒤 10에폭 동안 그보다 낮아지지 않아 55에폭에서 멈췄습니다.
- **`es_model`은 55에폭이 아니라 45에폭의 파라미터:** 되돌린 모델의 검증 손실이 `best_valid_loss`와 같습니다.
- **7절 표의 증강 + Dropout 0.5 행과 같은 값:** 6-3의 `combo_history`와 같은 시드로 학습해 최저 검증 손실(0.281)과 그 에폭(45)이 같습니다. 50에폭까지의 검증 손실 곡선은 7절 그래프의 증강 + Dropout 0.5 곡선과 같습니다.

## 9. PyTorch Lightning으로 다시 학습하기

8절에서 손으로 쓴 반복문(학습, 검증, 가장 좋은 파라미터 저장, 멈추기)을 교안 02 8절처럼 `Trainer`에 맡기고, 미니배치 하나에 할 일만 `LightningModule`에 적어 같은 설정(증강 + Dropout 0.5, patience 10)으로 학습합니다.

| 8절 손으로 쓴 코드 | Lightning |
|---|---|
| `train_one_epoch_augment` 안의 미니배치 한 번 | `training_step` |
| `evaluate` 안의 미니배치 한 번 | `validation_step` |
| `torch.optim.Adam(...)` | `configure_optimizers` |
| `model.train()`, `model.eval()` | `Trainer`가 알아서 바꿈 |
| `patience`, `best_epoch`, `break` | `EarlyStopping(monitor="val_loss", patience=10)` |
| `best_state = copy.deepcopy(...)` | `ModelCheckpoint(monitor="val_loss")`가 파일로 저장 |
| `es_rows.append(...)` | `self.log(...)`와 `CSVLogger` |
| `evaluate(model, test_loader, loss_fn)` | `test_step`과 `trainer.test(...)` |

아래 셀은 실행만 하세요. 교안 02처럼 Lightning이 실행마다 띄우는 안내 문구를 숨깁니다.

In [ ]:
import logging
import warnings

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

# Lightning이 실행마다 띄우는 안내 문구(장치 정보, 다른 서비스 소개, 성능 권고)를 숨기고 결과만 봅니다
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")

### 9-1. LightningModule 만들기

**배경**: 교안 02 8-3의 `LitCNN`과 같은 틀에서, 학습 미니배치만 증강하도록 `training_step`을 바꾼 클래스를 만듭니다.

**요구사항**
- (1) 골격의 `...` 일곱 곳을 채워 **`RpsClassifier`** 를 완성합니다. 클래스 문법은 골격의 주석을 따라 읽으면 됩니다.
- (2) `training_step`: 미니배치 그림을 `augment_rps`로 증강한 뒤 로짓을 구해 손실을 계산합니다(기록과 돌려주기는 골격에 있음).
- (3) `validation_step`, `test_step`: 증강 없이 로짓(`logits`)을 구해 손실을 계산합니다(정확도와 기록은 골격에 있음).
- (4) `configure_optimizers`: 이 모델의 파라미터(`self.parameters()`)를 학습률 0.001로 갱신하는 Adam 옵티마이저를 돌려줍니다.

**예시**: `self(X_batch)`는 `forward`를 불러 `self.model(X_batch)`를 돌려줍니다.

**확인 기준**: 안의 모델이 Dropout 0.5인 `make_rps_cnn`이고 그림 5장을 넣으면 `(5, 3)` 로짓이 나옵니다. `configure_optimizers`는 학습률 0.001인 Adam을 돌려주고, `validation_step`은 검증 그림을 그대로, `training_step`은 증강한 그림을 모델에 넣습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 8-3의 LitCNN과 메서드 이름, 순서가 같습니다. 다른 곳은 training_step의 증강 한 줄입니다.

세부구현:
1. 객체 자신을 함수처럼 부르면 forward가 실행되어 로짓을 돌려줍니다.
2. 손실은 __init__에서 붙여 둔 self.loss_fn에 로짓과 정답을 넘겨 구합니다.
3. test_step은 validation_step과 같은 계산이고 기록하는 이름만 test_로 시작합니다.
```

</details>

In [ ]:
# [작성] 미니배치 하나에 할 일을 적은 LightningModule을 완성하세요.
class RpsClassifier(L.LightningModule):  # L.LightningModule의 기능(학습 반복, 기록 등)을 물려받는 새 클래스
    """가위바위보 CNN을 Lightning 형식으로 감쌉니다."""

    def __init__(self, dropout_p):
        # 객체를 만들 때 한 번 실행됩니다. self는 지금 만들어지는 객체 자신입니다
        super().__init__()  # 물려받은 LightningModule의 준비 코드를 먼저 실행
        self.model = make_rps_cnn(dropout_p)  # 2절의 CNN을 객체 안에 보관
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, X):
        # self(X)로 부르면 실행됩니다
        return self.model(X)

    def training_step(self, batch, batch_idx):
        # Trainer가 학습 미니배치마다 부릅니다. 학습 모드는 Trainer가 켜 줍니다
        X_batch, y_batch = batch
        X_batch = ...
        loss = ...
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        return loss  # 돌려준 손실로 Trainer가 역전파와 갱신을 합니다

    def validation_step(self, batch, batch_idx):
        # Trainer가 에폭마다 검증 미니배치마다 부릅니다. 평가 모드와 no_grad는 Trainer가 켜 줍니다
        X_batch, y_batch = batch
        logits = ...
        loss = ...
        accuracy = (logits.argmax(dim=1) == y_batch).float().mean()
        self.log("val_loss", loss)
        self.log("val_acc", accuracy)

    def test_step(self, batch, batch_idx):
        # trainer.test가 테스트 미니배치마다 부릅니다
        X_batch, y_batch = batch
        logits = ...
        loss = ...
        accuracy = (logits.argmax(dim=1) == y_batch).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", accuracy)

    def configure_optimizers(self):
        # Trainer가 학습 시작 전에 한 번 부릅니다
        return ...

In [ ]:
# [자가채점] 모델 구조, 출력 모양, 옵티마이저, validation_step과 training_step이 모델에 넣는 그림을 확인합니다.
check_lit = RpsClassifier(dropout_p=0.5)
assert isinstance(check_lit.model, nn.Sequential) and check_lit.model[7].p == 0.5, "self.model은 make_rps_cnn(dropout_p)입니다."
check_lit.eval()
with torch.no_grad():
    check_logits = check_lit(X_valid[:5].to(device))
assert check_logits.shape == (5, 3), "forward는 self.model(X)를 돌려줍니다. 그림 5장이면 (5, 3)입니다."

check_optimizer = check_lit.configure_optimizers()
assert isinstance(check_optimizer, torch.optim.Adam) and check_optimizer.param_groups[0]["lr"] == 0.001, "configure_optimizers는 학습률 0.001인 Adam 옵티마이저를 돌려줍니다."

# validation_step이 검증 그림을 그대로 쓰는지: 평가 모드에서 모델에 들어간 그림을 기록해 원래 그림과 비교합니다
seen_batches = []
hook = check_lit.model.register_forward_hook(record_input)
check_lit.eval()
torch.manual_seed(0)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Trainer 밖에서 부르면 self.log가 경고를 내므로 이 확인에서만 숨깁니다
    with torch.no_grad():
        check_lit.validation_step((X_valid[:16].to(device), y_valid[:16].to(device)), 0)
hook.remove()
assert len(seen_batches) > 0 and torch.equal(seen_batches[-1], X_valid[:16]), "validation_step에서 모델이 받은 그림이 원래 검증 그림과 다릅니다. 검증 그림은 증강하지 않습니다. 증강은 training_step에만 넣으세요."

# training_step이 받은 그림을 증강하는지: 모델에 들어간 그림을 기록해 원래 그림과 비교합니다
seen_batches = []
hook = check_lit.model.register_forward_hook(record_input)
check_lit.train()
torch.manual_seed(0)
changed = 0
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Trainer 밖에서 training_step을 부르면 self.log가 경고를 내므로 이 확인에서만 숨깁니다
    for trial in range(4):
        check_batch = (X_train[:16].to(device), y_train[:16].to(device))
        check_loss = check_lit.training_step(check_batch, 0)
        if not torch.equal(seen_batches[-1], X_train[:16]):
            changed += 1
hook.remove()
assert isinstance(check_loss, torch.Tensor) and check_loss.dim() == 0, "training_step은 손실(숫자 하나인 텐서)을 돌려줍니다."
assert changed >= 2, "training_step에서 모델이 받은 그림이 원래 그림과 같습니다. 로짓을 구하기 전에 증강하세요."
print(f"✅ 9-1번 통과: 학습 입력이 증강된 횟수 {changed}/4")

**결과 읽기**
- **(5, 3):** `forward`가 그림 5장마다 세 로짓을 돌려줍니다.
- **검증 그림은 그대로, 학습 그림은 증강:** `validation_step`의 입력은 원래 검증 그림과 같고, `training_step`의 입력은 4번 가운데 4번 원래 그림과 달랐습니다.

### 9-2. Trainer로 학습하기

**배경**: 교안 02 8-4처럼 조기 종료, 체크포인트, 기록을 콜백과 로거로 붙여 `Trainer`가 8절의 반복문을 대신 돌리게 합니다.

**요구사항**
- (1) 시드를 0으로 고정(`L.seed_everything`)한 뒤 Dropout 0.5인 `RpsClassifier` **`lit_model`** 을 만듭니다.
- (2) `val_loss`를 보고 patience 10으로 멈추는 `EarlyStopping` **`early_stopping`** 을 만듭니다. **`checkpoint`**, **`logger`** 는 골격에 있습니다.
- (3) 세 부품을 넣은 `L.Trainer` **`trainer`** 를 만듭니다. 나머지 인자는 `max_epochs=60`, `accelerator=device.type`, `enable_progress_bar=False`, `enable_model_summary=False`, `deterministic="warn"`.
- (4) `trainer.fit`에 `lit_model`, 학습 DataLoader, 검증 DataLoader를 넘겨 학습합니다.

**예시**: 교안 02 8-4와 같은 순서(시드, 모델, 부품 세 개, Trainer, 학습)입니다.

**확인 기준**: `trainer`가 `max_epochs=60`이고 `early_stopping`(`val_loss`, patience 10), `checkpoint`, `logger`를 가집니다. 체크포인트에 저장된 가장 낮은 `val_loss`가 0.7보다 낮습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 8-4 셀과 같은 순서입니다(시드, 모델, 부품 세 개, Trainer, 학습).

세부구현:
1. 콜백 두 개는 callbacks 인자에 리스트로, 로거는 logger 인자에 넘깁니다.
2. fit은 모델, 학습 DataLoader, 검증 DataLoader 순서입니다.
```

</details>

In [ ]:
# [작성] Trainer에 조기 종료, 체크포인트, 기록을 붙여 학습하세요.

# 1. 시드 고정과 모델
...  # 시드를 0으로 고정
lit_model = ...

# 2. 조기 종료, 체크포인트(검증 손실이 가장 낮은 파라미터를 파일로 저장), 기록을 맡을 부품
early_stopping = ...
checkpoint = ModelCheckpoint(monitor="val_loss")
logger = CSVLogger(save_dir=material_dir / "output", name="rps_lightning")

# 3. Trainer
trainer = ...

# 4. 학습
...  # trainer로 lit_model을 학습(학습, 검증 DataLoader)
print("멈춘 에폭:", trainer.current_epoch, "| 가장 낮은 val_loss:", round(checkpoint.best_model_score.item(), 3))

In [ ]:
# [자가채점] 조기 종료와 체크포인트를 확인합니다.
assert isinstance(trainer, L.Trainer) and trainer.max_epochs == 60, "trainer는 max_epochs가 60인 L.Trainer입니다. `...`를 채웠는지 확인하세요."
assert lit_model.model[7].p == 0.5, "lit_model은 Dropout 0.5인 RpsClassifier입니다."
assert trainer.early_stopping_callback is early_stopping and early_stopping.patience == 10, "callbacks에 patience 10인 early_stopping을 넣으세요."
assert early_stopping.monitor == "val_loss", "EarlyStopping은 val_loss(검증 손실)를 봅니다."
assert trainer.checkpoint_callback is checkpoint and checkpoint.monitor == "val_loss", "callbacks에 골격의 checkpoint를 넣으세요."
assert trainer.logger is logger, "logger 인자에 골격의 logger를 넣으세요."
assert trainer.current_epoch > 10, "학습이 거의 진행되지 않았습니다. 마지막 `...` 자리에서 학습을 실행했는지 확인하세요."
assert checkpoint.best_model_score.item() < 0.7, f"가장 낮은 val_loss가 {checkpoint.best_model_score.item():.3f}입니다. 0.7보다 낮아야 합니다. training_step에서 증강했는지, Dropout이 0.5인지 확인하세요."
print("✅ 9-2번 통과")

아래 셀은 실행만 하세요. `CSVLogger`가 저장한 `metrics.csv`를 읽어 학습 곡선을 그립니다. 이 파일은 학습 손실 행과 검증 행이 따로 저장되어 있어 에폭마다 평균으로 한 행씩 합칩니다.

In [ ]:
# CSVLogger가 저장한 기록 읽기
metrics = pd.read_csv(Path(trainer.logger.log_dir) / "metrics.csv")
lit_history = metrics.groupby("epoch")[["train_loss", "val_loss", "val_acc"]].mean().reset_index()
lit_history["epoch"] = lit_history["epoch"] + 1  # Lightning은 에폭을 0부터 셉니다
display(lit_history.set_index("epoch").tail().round(3))

# 학습 곡선
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(lit_history["epoch"], lit_history["train_loss"], marker="o", markersize=3, label="train_loss")
ax.plot(lit_history["epoch"], lit_history["val_loss"], marker="o", markersize=3, label="val_loss")
best_lit_epoch = int(lit_history.loc[lit_history["val_loss"].idxmin(), "epoch"])
ax.axvline(best_lit_epoch, color="green", linestyle="--", label=f"가장 좋았던 에폭 {best_lit_epoch}")
ax.set_title("Lightning으로 학습한 증강 + Dropout 0.5", fontsize=14)
ax.set_xlabel("에폭", fontsize=12)
ax.set_ylabel("손실", fontsize=12)
ax.tick_params(labelsize=11)
ax.legend(loc="upper right", fontsize=11)
plt.tight_layout()
plt.show()

**결과 읽기**
- **42에폭에서 멈춤, 가장 낮은 `val_loss` 0.331(32에폭):** 8절은 45에폭의 0.281이었습니다. 설정은 같지만 미니배치 순서와 증강에 쓰는 난수를 뽑는 순서가 달라 곡선이 똑같지는 않습니다.
- **직접 쓰지 않은 코드:** 반복문, 모드 전환, 조기 종료, 파라미터 저장을 `Trainer`가 대신했습니다.

## 10. 테스트 데이터로 한 번 평가하기

**배경**: 설정 고르기와 멈출 때 정하기를 모두 검증 데이터로 끝냈으므로, 이제 학습과 검증에 쓰지 않은 테스트 그림 372장으로 두 모델을 한 번씩 평가합니다(이 값을 보고 설정을 다시 바꾸면 테스트 데이터도 고르는 데 쓴 셈이 됩니다).

**요구사항**
- (1) `evaluate`로 `es_model`의 테스트 손실 **`es_test_loss`** 를 만듭니다.
- (2) 같은 호출의 정확도 **`es_test_accuracy`** 를 만듭니다.
- (3) 교안 02 9-5(함께 따라하기, 완성 코드는 8-6의 `trainer.validate`가 같은 인자)처럼 `trainer.test`에 `lit_model`, 테스트 DataLoader(`dataloaders`), 가장 좋은 체크포인트(`ckpt_path="best"`), `verbose=False`를 넘긴 결과 **`test_results`** 를 만듭니다.
- (4) `test_results`에서 꺼낸 테스트 손실 **`lit_test_loss`**, 정확도 **`lit_test_accuracy`** 를 만듭니다.
- (5) 두 모델의 `모델`, `테스트 손실`, `테스트 정확도` 세 키의 dict를 모은 표 **`test_table`**(2행)을 만듭니다.

**예시**: `test_results`는 딕셔너리 하나가 든 리스트이고, 키는 `test_step`에서 기록한 `test_loss`, `test_acc`입니다.

**확인 기준**: 두 모델 모두 테스트 정확도가 0.75보다 높습니다(무작위로 고를 때 0.33). 네 값이 각 모델을 테스트 그림 전체로 다시 평가한 값과 거의 같고, `lit_model`의 파라미터가 체크포인트 파일(검증 손실이 가장 낮았던 에폭)의 파라미터와 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 손으로 쓴 모델은 evaluate로, Lightning 모델은 test_step을 부르는 trainer.test로 평가합니다. 둘 다 테스트 DataLoader를 씁니다.

세부구현:
1. trainer.test의 테스트 DataLoader는 dataloaders 인자로 넘깁니다.
2. test_results의 첫 원소가 딕셔너리이고, 키는 test_step에서 기록한 이름입니다.
3. 표의 한 행은 모델, 테스트 손실, 테스트 정확도 세 키의 dict입니다.
```

</details>

In [ ]:
# [작성] 조기 종료 모델과 Lightning 모델을 테스트 데이터로 한 번 평가하세요.

# 1. 손으로 쓴 조기 종료 모델
es_test_loss, es_test_accuracy = ...

# 2. Lightning 모델: 가장 좋은 체크포인트를 불러와 test_step으로 평가
test_results = ...
lit_test_loss = ...
lit_test_accuracy = ...

# 3. 두 모델의 테스트 결과
test_table = pd.DataFrame([
    ...,  # 조기 종료 모델의 행(dict)
    ...,  # Lightning 모델의 행(dict)
])
display(test_table.round(3))

In [ ]:
# [자가채점] 네 값을 테스트 데이터로 다시 계산해 비교하고, 불러온 체크포인트를 확인합니다.
assert isinstance(test_table, pd.DataFrame) and list(test_table.columns) == ["모델", "테스트 손실", "테스트 정확도"] and len(test_table) == 2, "test_table은 모델, 테스트 손실, 테스트 정확도 세 열의 2행입니다. `...`를 채웠는지 확인하세요."

# es_model에 테스트 그림 372장을 한 번에 넣어 손실과 정확도를 따로 계산해 비교합니다
es_model.eval()
with torch.no_grad():
    es_logits = es_model(X_test.to(device)).cpu()
recheck_es_loss = nn.functional.cross_entropy(es_logits, y_test).item()
recheck_es_accuracy = (es_logits.argmax(dim=1) == y_test).float().mean().item()
assert abs(es_test_loss - recheck_es_loss) < 0.005 and abs(es_test_accuracy - recheck_es_accuracy) < 0.01, "es_test_loss, es_test_accuracy가 es_model을 테스트 데이터로 평가한 값과 다릅니다. 검증 DataLoader가 아니라 테스트 DataLoader를 넘기세요."

saved_state = torch.load(checkpoint.best_model_path, map_location="cpu", weights_only=False)["state_dict"]
assert torch.equal(lit_model.model[8].weight.detach().cpu(), saved_state["model.8.weight"]), "lit_model의 파라미터가 가장 좋은 체크포인트 파일과 다릅니다. trainer.test에 가장 좋은 체크포인트를 쓰라는 인자를 넘겼는지 확인하세요."
# 불러온 체크포인트 모델도 같은 방법으로 따로 계산해 비교합니다
lit_model.to(device)
lit_model.eval()
with torch.no_grad():
    lit_logits = lit_model(X_test.to(device)).cpu()
recheck_lit_loss = nn.functional.cross_entropy(lit_logits, y_test).item()
recheck_lit_accuracy = (lit_logits.argmax(dim=1) == y_test).float().mean().item()
assert abs(lit_test_loss - recheck_lit_loss) < 0.005 and abs(lit_test_accuracy - recheck_lit_accuracy) < 0.01, f"lit_test_loss, lit_test_accuracy({lit_test_loss:.4f}, {lit_test_accuracy:.4f})가 체크포인트 모델을 테스트 데이터로 평가한 값({recheck_lit_loss:.4f}, {recheck_lit_accuracy:.4f})과 다릅니다. trainer.test에 테스트 DataLoader를 넘겼는지, test_step이 손실과 정확도를 test_loss, test_acc로 기록하는지 확인하세요."
assert es_test_accuracy > 0.75 and lit_test_accuracy > 0.75, f"테스트 정확도가 {es_test_accuracy:.3f}, {lit_test_accuracy:.3f}입니다. 0.75보다 높아야 합니다. 8절, 9절이 증강 + Dropout 0.5로 학습했는지 확인하세요."
print("✅ 10번 통과")

**결과 읽기**
- **테스트 정확도 0.866, 0.839:** 조기 종료 모델이 0.866, Lightning 모델이 0.839입니다. 검증 데이터(손 모델 5, 6)로 고른 설정이 학습과 검증에 쓰지 않은 테스트 그림에서도 통했습니다. 테스트는 마지막 확인이므로 두 모델의 이 차이로 모델을 고르지 않습니다.
- **검증 정확도 0.554에서 0.867:** 같은 검증 데이터끼리 비교하면, 검증 손실이 가장 낮았던 에폭의 검증 정확도가 규제 없음(10에폭)에서 증강(31에폭)으로 올랐습니다. 학습 그림에 없는 변화(손의 방향, 위치)를 증강으로 보여 준 것이 손 모델이 적은 문제에 가장 잘 맞았습니다.